# ▶ Architecture 4 (MLA) trained from scratch on ~1.5B tokens (2 sessions)
Same MLA architecture as arch 4 (≈88M params), **trained from scratch** for 45,000 steps × 32,768 tokens ≈ **1.47B tokens** (arch 4 saw 393M). Data = `data-sbpe8000` (393M, balanced) + `data-sbpe8000-slice2` (590M, unseen part of the files), sampled **50/50** → ≈17% acts / 25% SC / 28% HC / 30% tribunals; neither set contains the validation documents. Both GPUs (DataParallel), EMA 0.999, warm-up 1000, cosine 3e-4 → 3e-5.

**Session 1:** GPU T4 x2, Secret `HF_TOKEN`, Add Input → both data datasets. Save & Run All (Commit). Training runs until ~10.7 h of session time (~30k steps), then stops by itself and still produces a (partial-model) submission + bundle.
Then create a Dataset (e.g. `arch4-big-s1`) from that version's Output folder `runs/arch4_mla_big/` (it must contain **ckpt_last.pt**).

**Session 2:** same notebook, add that dataset too, set `resume_from = "/kaggle/input/.../ckpt_last.pt"` and commit. It prints `Resumed from … at step ~30,000`, finishes the remaining ~15k steps (~5 h), then decoding search, generation, zip.

Compare the final val loss with arch 4's **1.916** (same validation set). Total GPU ≈ 16–17 h.


# Phase 1 — Legal Language Model (from scratch) · v2
**CS781 Capstone · Kaggle GPU notebook · Arch #1 decoder-only (`arch="decoder"`) · Arch #2 encoder-decoder (`arch="encdec"`) · Arch #3 prefix-LM (`arch="prefixlm"`) · Arch #4 MLA (`arch="mla"`)**

End-to-end pipeline for one Phase-1 experiment:

1. Stream the gated `Exploration-Lab/CS-781-Capstone` Phase-1 data (no 19 GB download).
2. Split **by document** into train / held-out validation, and build a **local dev set** (512-word prompt → 128-word reference) from the held-out documents.
3. Train **your A1 byte-level BPE** tokenizer, encode the corpus in parallel, save `train.bin` / `val.bin`, and reuse them in every architecture notebook (same data, so the comparison is controlled).
4. Build **your A3 DecoderLM** (50M–300M params), train it from scratch with resumable checkpoints.
5. Evaluate locally (val loss/perplexity + ROUGE-L / BLEU-4 / BERTScore on the dev set) and try different decoding settings.
6. Generate exactly 128 words for each official test prompt, check the output, and write `prediction.json` → `submission.zip`.
7. Log the run to `experiments.jsonl` and export the checkpoint + code for Phases 2–7.

---
### What changed from draft-1 (and why) — you must be able to explain each of these in the viva
| # | Problem seen in draft-1 output | Fix |
|---|---|---|
| FIX-1 | Smoke-test val loss was **15.2 → 10.4**, which is *higher* than `ln(2000)=7.6` (the loss of a uniform guess). Cause: no weight init. `nn.Embedding` defaults to N(0,1) and is **tied** to the output head, so initial logits have std ≈ √d_model ≈ 16, which makes the loss very large. That is why the output was "the the the …". | GPT-style init N(0, 0.02) for Linear/Embedding, plus a check that the initial loss is ≈ ln(V). |
| FIX-2 | Once embeddings are N(0, 0.02), the token signal is ~50× smaller than the sinusoidal position signal (amplitude ~1). | Multiply token embeddings by √d_model, as in the original Transformer. One marked line in `model_arch.py`, controlled by `scale_emb`. |
| FIX-3 | A3 default is **post-LN** (`norm_first=False`). Post-LN often trains unstably at 10+ layers without very careful warmup (Xiong et al. 2020). | `norm_first=True` (pre-LN). This flag already exists in your A3 code. |
| FIX-4 | Word counts were **78–80**, but Phase 1 needs 128 words. Generation used a fixed token cap and also stopped at `</s>`. | Generate until ≥129 words, then cut to 128. Special tokens are masked during sampling. |
| FIX-5 | Generating one prompt at a time took 12 min even for the tiny model. With the real model it would take hours. | Batched generation. Prompts in a batch are cut to the same token length, so no padding mask is needed. With **GPU T4 x2**, batches are split across **both GPUs** (about 2x faster; identical output). Progress prints at least every 5 min. |
| FIX-6 | `hf_hub_download` downloaded all ~19 GB of training JSON even in smoke mode. | Stream with `HfFileSystem`, so only the bytes you use are read. |
| FIX-7 | The config used 20M train tokens but consumed 4000×32×1024 ≈ 131M tokens, so the same data was seen **~6.5 times**. Validation was the token tail of the stream (train and val could share a document). | Split by document, **stratified per source** (equal val tokens and dev prompts from each of the 4 sources; val loss also reported per source). The data budget (`train_tokens`) is a **fixed** number (= what arch-1 consumes), independent of each architecture's steps/batch, so every architecture gets the *identical* split and tokenizer. A tokenizer fingerprint is checked on resume. |
| FIX-8 | Validation used 40 *random* batches, so it was noisy and not comparable across architectures. | Fixed, evenly spaced validation windows (the same for every run). |
| FIX-9 | Only the best weights were saved. A Kaggle timeout or disconnect lost the whole run. | Resumable `ckpt_last.pt` (model + optimizer + scaler + step + RNG), plus a **session clock**: training stops early enough that generation + zip always finish inside Kaggle's 12 h limit. Test predictions are written *before* the optional dev evaluation. |
| FIX-10 | No local evaluation (the document, Sec 16, asks for it). | Local dev set built from held-out documents, scored with ROUGE-L, BLEU-4 and BERTScore. |
| misc | Weight decay was also applied to LayerNorm and bias. LR decayed all the way to 0. The prompt could end mid-word. | Separate no-decay parameter group, `min_lr_ratio`, and a trailing space after the prompt. |

### How to run on Kaggle
1. Settings → Accelerator **GPU T4 x2** (training uses one GPU, generation uses both), Internet **On**. Add the Secret `HF_TOKEN`.
2. `SMOKE_TEST=True` first (~15 min). Check: initial loss ≈ ln(V), word counts = 128, zip passes the checks.
3. `SMOKE_TEST=False` → **Save Version → Save & Run All (Commit)**. Commit mode runs in the background for up to 12 h.
4. When it finishes, create a **Kaggle Dataset from the output**. Point `prebuilt_dir` at its `data_*` folder in the other architecture notebooks, and `resume_from` at `ckpt_last.pt` if a run was cut off.

In [ ]:
# 1 · Install dependencies (torch is already on Kaggle). Requires Internet = ON.
!pip -q install -U "huggingface_hub>=0.23" ijson rouge-score sacrebleu bert-score

In [ ]:
# 2 · Configuration ------------------------------------------------------------
RUN_NAME   = "arch4_mla_big"                 # names the run folder, checkpoints, logs and zip
SMOKE_TEST = False                       # True = tiny end-to-end check; False = real run

CONFIG = dict(
    hf_repo   = "Exploration-Lab/CS-781-Capstone",
    phase_dir = "Phase_1_Pretraining",
    sources   = ["acts", "high_courts", "supreme_court", "tribunals"],
    mix       = {"acts": 1, "high_courts": 1, "supreme_court": 1, "tribunals": 1},  # relative char-budget per source (a data-mixing experiment)
    prebuilt_dir = "/kaggle/input/datasets/iamangelo/data-sbpe8000",        # e.g. "/kaggle/input/<your-dataset>/data_bpe8000" -> reuse tokenizer + tokens (skip steps 5-6)

    # ---- tokenizer ----
    tokenizer_type  = "space",     # "a1" = your A1 BPE (every space is its own token, ~2.3 tokens/word)
                                # "space" = space-attached BPE (" the", ~1.3 tokens/word -> whole prompt fits)
    vocab_size      = 8000,
    tok_train_chars = 30_000_000,

    # ---- data budget / validation ----
    train_tokens   = 393_217_025,  # FIXED data budget (= 4000 steps x 32 x 1024 + 1025, what arch-1 uses).
                                   # Do NOT change between architectures: it defines the train/val split.
    val_tokens     = 2_000_000,
    val_doc_frac   = 0.02,      # fraction of documents per source held out (never trained on)
    n_dev_examples = 100,       # local dev prompts built from held-out docs
    skip_chars_per_source = 0,  # data builder: skip this many characters at the start of each source (fresh slice)
    tokenizer_from = None,      # data builder: reuse this dataset's tokenizer + val/dev sets (continued training)
    extra_train_dirs = ["/kaggle/input/datasets/iamangelo/data-sbpe8000-skip-481m"],      # more prebuilt datasets (same tokenizer) whose train.bin is trained on as well
    train_mix = [0.5, 0.5],             # sampling weights for [prebuilt_dir, *extra_train_dirs]; [] = proportional to size

    # ---- model (your A3 DecoderLM) ----
    d_model=768, n_layers=12, n_heads=12, n_kv_heads=12, d_ff=3072,
    max_seq_len=1024, dropout=0.0, pos_emb_type="sinusoidal",
    norm_first = True,          # [FIX-3] pre-LN
    scale_emb  = True,          # [FIX-2] sqrt(d_model) token-embedding scale

    # ---- architecture switch ----
    arch = "mla",               # "decoder" = arch-1 | "encdec" = arch-2 | "prefixlm" = arch-3 | "mla" = arch-4
    # arch-2 only (uses d_model, n_heads, d_ff, dropout above):
    enc_layers=6, dec_layers=4, # encoder / decoder depth
    enc_len=768, dec_len=384,   # window split: encoder reads 768 prompt tokens, decoder predicts 384
    # arch-3 only (uses d_model, n_layers, n_heads, n_kv_heads, max_seq_len, dropout above):
    ffn_hidden=2048,            # SwiGLU hidden size
    prefix_min=0.0, prefix_max=0.85,  # training prefix length p ~ U[prefix_min, prefix_max] * max_seq_len
    eval_prefix=0.75,           # fixed prefix fraction for the validation loss
    # arch-4 only (uses d_model, n_layers, n_heads, ffn_hidden, max_seq_len, dropout above):
    mla_kv_rank=256,            # compressed key/value latent per token (what the KV cache stores)
    mla_nope_dim=64, mla_rope_dim=32, mla_v_dim=64,   # per-head dims: content / decoupled-RoPE / value

    # ---- training ----
    batch_size=8, grad_accum=4, lr=3e-4, min_lr_ratio=0.1, weight_decay=0.1,
    warmup_steps=1000, max_steps=40000, grad_clip=1.0,
    eval_every=2500, eval_batches=40, ckpt_every=500,
    session_limit_hours = 6.8, # Kaggle kills a session at 12 h (counted from the FIRST cell, incl. data prep)
    reserve_hours_after_training = 0.6,  # time kept for test generation (~2-2.5 h) + zip + dev eval
    resume_from = "/kaggle/input/datasets/iamangelo/arch4-mla-big-2/ckpt_last (4).pt",         # path to a previous ckpt_last.pt (e.g. /kaggle/input/.../ckpt_last.pt)
    init_from = None,           # continue from these weights with a NEW schedule (e.g. more steps on a fresh data slice)
    init_load_optimizer = True, # with init_from: also restore Adam moments + grad scaler (needs a ckpt_last.pt)
    train_gpus = None,          # None = all visible GPUs (DataParallel: ~1.7x faster on T4 x2); 1 = single GPU
    ema_decay = 0.999,            # >0 (e.g. 0.999): also track an EMA of the weights; the better of raw/EMA is saved

    # ---- generation ----
    gen_words=128, max_new_tokens=1024, gen_batch_size=16, gen_ctx_tokens=None,  # cap only; stops early at 128 words
    temperature=0.8, top_k=50, top_p=0.95, repetition_penalty=1.0,             # 1.0 = off
    gen_gpus = None,            # None = use every visible GPU for generation (Kaggle 'GPU T4 x2' -> 2x faster)
    run_bertscore = True,       # local BERTScore on the dev set (downloads roberta-large once)
    gen_reserve_tokens = 256,   # prefix-LM: keep at most max_seq_len - this prompt tokens (room for the KV cache)
    decode_search = False,       # True: try decode_grid on dev prompts BEFORE the test set, use the best BERTScore
    decode_search_n = 60,       # dev prompts per decoding setting (balanced over the 4 sources)
    mbr_samples = 0,            # >1 = MBR decoding: sample this many continuations per prompt, keep the consensus one
    mbr_auto = True,            # compare MBR vs a single sample on dev prompts first; use MBR only if it scores higher
    ensemble_ckpts = [],        # other trained checkpoints (SAME tokenizer, KV-cache archs: prefixlm / mla) to ensemble
    ensemble_weight_grid = [[0.5, 0.5], [0.65, 0.35], [0.35, 0.65]],  # weights for [this model, *ensemble_ckpts];
                                # the dev check compares each against this model alone and keeps the best
    decode_grid = [
        dict(name="t0.8_p0.95",        temperature=0.8, top_k=50, top_p=0.95, repetition_penalty=1.0),
        dict(name="t0.7_p0.95",        temperature=0.7, top_k=50, top_p=0.95, repetition_penalty=1.0),
        dict(name="t0.6_p0.9",         temperature=0.6, top_k=50, top_p=0.90, repetition_penalty=1.0),
        dict(name="t0.7_p0.95_rep1.1", temperature=0.7, top_k=50, top_p=0.95, repetition_penalty=1.1),
        dict(name="t0.6_p0.9_rep1.15", temperature=0.6, top_k=50, top_p=0.90, repetition_penalty=1.15),
        dict(name="greedy_rep1.2",     temperature=0.0, top_k=0,  top_p=1.0,  repetition_penalty=1.2),
    ],

    seed=1337, out_dir="/kaggle/working",
)

if SMOKE_TEST:
    CONFIG.update(
        vocab_size=2000, tok_train_chars=1_000_000, train_tokens=246_017, val_tokens=50_000, val_doc_frac=0.05, n_dev_examples=16,
        d_model=256, n_layers=4, n_heads=4, n_kv_heads=4, d_ff=1024, max_seq_len=256,
        enc_layers=2, dec_layers=2, enc_len=128, dec_len=128, ffn_hidden=688,
        gen_reserve_tokens=96, decode_search_n=8, mla_kv_rank=64,
        batch_size=8, grad_accum=1, warmup_steps=20, max_steps=120,
        eval_every=40, eval_batches=10, ckpt_every=40, run_bertscore=False,
    )

import os, random, math, json, time, pickle, zipfile, shutil
SESSION_START = time.time()                      # the 12 h Kaggle clock starts (roughly) here
def session_hours(): return (time.time() - SESSION_START) / 3600
from dataclasses import asdict
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
random.seed(CONFIG["seed"]); np.random.seed(CONFIG["seed"]); torch.manual_seed(CONFIG["seed"])
device = "cuda" if torch.cuda.is_available() else "cpu"

OUT_ROOT = CONFIG["out_dir"]
RUN_DIR  = os.path.join(OUT_ROOT, "runs", RUN_NAME + ("_smoke" if SMOKE_TEST else ""))
DATA_DIR = CONFIG["prebuilt_dir"] or os.path.join(
    OUT_ROOT, f"data_{'sbpe' if CONFIG['tokenizer_type'] == 'space' else 'bpe'}{CONFIG['vocab_size']}"
    + (f"_skip{CONFIG['skip_chars_per_source'] // 10**6}M" if CONFIG["skip_chars_per_source"] else "")
    + ("_smoke" if SMOKE_TEST else ""))
os.makedirs(RUN_DIR, exist_ok=True)
SEQ_LEN = CONFIG["enc_len"] + CONFIG["dec_len"] if CONFIG["arch"] == "encdec" else CONFIG["max_seq_len"]
TOKENS_PER_STEP = CONFIG["batch_size"] * CONFIG["grad_accum"] * SEQ_LEN
TRAIN_TOKENS = CONFIG["train_tokens"]
print("Device:", device, "| SMOKE_TEST:", SMOKE_TEST, "| run dir:", RUN_DIR, "| data dir:", DATA_DIR)
print(f"Tokens per optimizer step: {TOKENS_PER_STEP:,} | tokens the run will consume: {CONFIG['max_steps']*TOKENS_PER_STEP:,}")

In [ ]:
# 3 · Hugging Face login (the dataset is gated) --------------------------------
# Accept the dataset terms once at https://huggingface.co/datasets/Exploration-Lab/CS-781-Capstone
from huggingface_hub import login
token = None
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    token = user_secrets.get_secret("HF_TOKEN")
    print("Loaded HF token from Kaggle Secrets.")
except Exception:
    pass
if not token:
    import getpass
    token = os.environ.get("HF_TOKEN") or getpass.getpass("Enter your Hugging Face token: ")
login(token=token)
print("Logged in to Hugging Face.")

## 4 · Your Assignment code (A3 model + A1 tokenizer)
`rope.py`, `attention.py` and `a1_tokenizer.py` are your code, unchanged. `model_arch.py` has one marked change (**[FIX-2]**: a `scale_emb` flag, off by default).

In [ ]:
import os
os.makedirs('a3src', exist_ok=True)

In [ ]:
%%writefile a3src/rope.py
# TODO 3.1: rotary positional embeddings from scratch (split-half convention).
import torch


def build_rope_cache(max_seq_len, d_head, base=10000.0):
    assert d_head % 2 == 0, "RoPE requires an even head dimension"
    inv_freq = 1.0 / (base ** (torch.arange(0, d_head, 2).float() / d_head))   # (d_head/2,)
    positions = torch.arange(max_seq_len).float()
    angles = torch.outer(positions, inv_freq)                                  # (T, d_head/2)
    angles = torch.cat([angles, angles], dim=-1)                               # (T, d_head)
    return angles.cos(), angles.sin()


def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)


def apply_rope(x, cos, sin):
    return x * cos + rotate_half(x) * sin

In [ ]:
%%writefile a3src/attention.py
# TODO 3.2: MHA / GQA / MQA attention with optional RoPE applied to q and k.
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from rope import build_rope_cache, apply_rope


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.0,
                 use_rope=False, attention_scale=None):
        super().__init__()
        assert d_model % n_heads == 0 and n_heads % n_kv_heads == 0
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.n_rep = n_heads // n_kv_heads
        self.d_head = d_model // n_heads
        self.use_rope = use_rope
        if attention_scale is None:
            self.attention_scale = 1.0 / math.sqrt(self.d_head)
        else:
            self.attention_scale = float(attention_scale)

        self.q_proj = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.out_proj = nn.Linear(n_heads * self.d_head, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", mask, persistent=False)

        if use_rope:
            cos, sin = build_rope_cache(max_seq_len, self.d_head)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)

    def forward(self, x):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)

        if self.use_rope:
            q = apply_rope(q, self.rope_cos[:T], self.rope_sin[:T])
            k = apply_rope(k, self.rope_cos[:T], self.rope_sin[:T])

        k = k.repeat_interleave(self.n_rep, dim=1)
        v = v.repeat_interleave(self.n_rep, dim=1)

        scores = torch.matmul(q, k.transpose(-2, -1)) * self.attention_scale
        scores = scores.masked_fill(~self.causal_mask[:T, :T], float("-inf"))
        attn = self.attn_dropout(F.softmax(scores, dim=-1))
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(B, T, self.n_heads * self.d_head)
        return self.resid_dropout(self.out_proj(out))

In [ ]:
%%writefile a3src/model_arch.py
# TODO 3.2: pass use_rope through the decoder; only build sinusoidal PE when requested.
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F
from attention import CausalSelfAttention


@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 256
    n_layers: int = 4
    n_heads: int = 4
    n_kv_heads: int = 4
    d_ff: int = 1024
    max_seq_len: int = 256
    dropout: float = 0.1
    pos_emb_type: str = "sinusoidal"   # "sinusoidal" or "rope"
    attention_scale: float = None
    norm_first: bool = False
    scale_emb: bool = False            # [FIX-2] multiply token embeddings by sqrt(d_model) (Vaswani et al. 2017, Sec 3.4)


def sinusoidal_positional_encoding(max_len, d_model):
    pe = torch.zeros(max_len, d_model)
    position = torch.arange(0, max_len).unsqueeze(1).float()
    div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm_first = cfg.norm_first
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.n_kv_heads, cfg.max_seq_len,
                                        cfg.dropout, use_rope=(cfg.pos_emb_type == "rope"),
                                        attention_scale=cfg.attention_scale)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x):
        if self.norm_first:
            x = x + self.attn(self.ln1(x))
            x = x + self.ffn(self.ln2(x))
        else:
            x = self.ln1(x + self.attn(x))
            x = self.ln2(x + self.ffn(x))
        return x


class DecoderLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        if cfg.pos_emb_type == "sinusoidal":
            pe = sinusoidal_positional_encoding(cfg.max_seq_len, cfg.d_model)
            self.register_buffer("pos_emb", pe, persistent=False)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight

    def forward(self, idx):
        B, T = idx.shape
        x = self.tok_emb(idx)
        if self.cfg.scale_emb:             # [FIX-2]
            x = x * math.sqrt(self.cfg.d_model)
        if self.cfg.pos_emb_type == "sinusoidal":
            x = x + self.pos_emb[:T].unsqueeze(0)
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        return self.head(x)

In [ ]:
%%writefile a3src/encdec.py
# Architecture 2: Encoder-decoder Transformer for "prompt -> continuation" language modelling.
#
# Every training window of enc_len + dec_len contiguous tokens is split in two:
#   encoder  <- first enc_len tokens (the "prompt"), bidirectional self-attention
#   decoder  <- next dec_len tokens (the "continuation"), causal self-attention + cross-attention
# The loss is computed on the continuation tokens only, which is exactly the Phase-1 test task.
# Positions: RoPE (your A3 rope.py) inside self-attention; no positional signal in cross-attention.
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F
from rope import build_rope_cache, apply_rope


@dataclass
class EncDecConfig:
    vocab_size: int
    d_model: int = 768
    n_heads: int = 12
    d_ff: int = 3072
    enc_layers: int = 6
    dec_layers: int = 4
    enc_len: int = 768           # prompt tokens the encoder reads
    dec_len: int = 384           # continuation tokens the decoder predicts per window
    dropout: float = 0.1
    bos_id: int = 2              # decoder start token (<s> in your A1 tokenizer)

    @property
    def max_seq_len(self):       # length of one training window (used by the data pipeline)
        return self.enc_len + self.dec_len


class Attention(nn.Module):
    """Multi-head attention. Self-attention when context is None (optionally with RoPE),
    cross-attention when context is given (queries from x, keys/values from context)."""

    def __init__(self, d_model, n_heads, dropout, rope_len=None):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads, self.d_head, self.dropout = n_heads, d_model // n_heads, dropout
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.kv_proj = nn.Linear(d_model, 2 * d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.use_rope = rope_len is not None
        if self.use_rope:
            cos, sin = build_rope_cache(rope_len, self.d_head)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)

    def forward(self, x, context=None, causal=False):
        B, T, C = x.shape
        src = x if context is None else context
        S = src.size(1)
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)          # (B,H,T,Dh)
        k, v = self.kv_proj(src).view(B, S, 2, self.n_heads, self.d_head).permute(2, 0, 3, 1, 4)
        if self.use_rope:
            # cast the cache to q's dtype so q, k, v share one dtype under fp16 autocast
            q = apply_rope(q, self.rope_cos[:T].to(q.dtype), self.rope_sin[:T].to(q.dtype))
            k = apply_rope(k, self.rope_cos[:S].to(k.dtype), self.rope_sin[:S].to(k.dtype))
        out = F.scaled_dot_product_attention(q, k, v, is_causal=causal,
                                             dropout_p=self.dropout if self.training else 0.0)
        return self.out_proj(out.transpose(1, 2).reshape(B, T, C))


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model),
                                 nn.Dropout(dropout))

    def forward(self, x):
        return self.net(x)


class EncoderLayer(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = Attention(cfg.d_model, cfg.n_heads, cfg.dropout, rope_len=cfg.enc_len)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x):                                   # pre-LN, bidirectional
        x = x + self.drop(self.attn(self.ln1(x)))
        return x + self.ffn(self.ln2(x))


class DecoderLayer(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.self_attn = Attention(cfg.d_model, cfg.n_heads, cfg.dropout, rope_len=cfg.dec_len)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.cross_attn = Attention(cfg.d_model, cfg.n_heads, cfg.dropout)          # no RoPE
        self.ln3 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, y, memory):                           # pre-LN
        y = y + self.drop(self.self_attn(self.ln1(y), causal=True))
        y = y + self.drop(self.cross_attn(self.ln2(y), context=memory))
        return y + self.ffn(self.ln3(y))


class EncDecLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.emb = nn.Embedding(cfg.vocab_size, cfg.d_model)       # shared by encoder, decoder, head
        self.emb_drop = nn.Dropout(cfg.dropout)
        self.encoder = nn.ModuleList([EncoderLayer(cfg) for _ in range(cfg.enc_layers)])
        self.enc_norm = nn.LayerNorm(cfg.d_model)
        self.decoder = nn.ModuleList([DecoderLayer(cfg) for _ in range(cfg.dec_layers)])
        self.dec_norm = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.emb.weight                          # weight tying

    def encode(self, src):
        """src (B, S<=enc_len) -> encoder hidden states (B, S, d). Also the Phase-2 feature extractor."""
        x = self.emb_drop(self.emb(src))
        for layer in self.encoder:
            x = layer(x)
        return self.enc_norm(x)

    def decode(self, memory, dec_in):
        """memory (B, S, d), dec_in (B, T<=dec_len) -> next-token logits (B, T, V)."""
        y = self.emb_drop(self.emb(dec_in))
        for layer in self.decoder:
            y = layer(y, memory)
        return self.head(self.dec_norm(y))

    def split(self, x):
        """Contiguous window x (B, enc_len+dec_len) -> encoder input, decoder input, decoder targets."""
        E = self.cfg.enc_len
        src, tgt = x[:, :E], x[:, E:]
        bos = torch.full((x.size(0), 1), self.cfg.bos_id, dtype=x.dtype, device=x.device)
        dec_in = torch.cat([bos, tgt[:, :-1]], dim=1)               # teacher forcing, shifted right
        return src, dec_in, tgt

    def forward(self, x):
        src, dec_in, _ = self.split(x)
        return self.decode(self.encode(src), dec_in)

    def compute_loss(self, x, y=None):
        """Cross-entropy on the continuation (decoder) tokens only. y is unused: targets come from x."""
        src, dec_in, tgt = self.split(x)
        logits = self.decode(self.encode(src), dec_in)
        return F.cross_entropy(logits.float().reshape(-1, logits.size(-1)), tgt.reshape(-1))

In [ ]:
%%writefile a3src/prefixlm.py
# Architecture 3: Prefix-LM (hybrid encoder/decoder in ONE stack) with RMSNorm, SwiGLU, RoPE, KV cache.
#
# One Transformer stack, two attention patterns decided by a prefix length p:
#   * prefix tokens   [0, p)  attend to EVERY prefix token (bidirectional, like an encoder)
#   * continuation    [p, T)  attend causally to everything before them (like a decoder)
#   mask: token i may attend to token j  <=>  (j <= i) or (j < p)
# Loss: only on predictions that cannot see their own target, i.e. positions i >= p-1.
# p is sampled per sequence during training (p = 0 -> plain causal LM), so the model learns both
# "read a prompt bidirectionally, then continue" and ordinary language modelling.
# Generation: the prompt is the prefix (one bidirectional "prefill" pass that fills a KV cache),
# then each new token is a single-position step that reuses the cached keys/values.
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F
from rope import build_rope_cache, apply_rope


@dataclass
class PrefixLMConfig:
    vocab_size: int
    d_model: int = 768
    n_layers: int = 12
    n_heads: int = 12
    n_kv_heads: int = 12          # < n_heads -> grouped-query attention (smaller KV cache)
    ffn_hidden: int = 2048        # SwiGLU hidden size (~8/3 * d_model keeps params like a 4x GELU FFN)
    max_seq_len: int = 1024
    dropout: float = 0.0
    prefix_min: float = 0.0       # training prefix length p ~ U[prefix_min*T, prefix_max*T]
    prefix_max: float = 0.85
    eval_prefix: float = 0.75     # fixed p for validation loss (deterministic, task-like)


class RMSNorm(nn.Module):
    """x / sqrt(mean(x^2) + eps) * g  - no mean subtraction, no bias (cheaper than LayerNorm)."""

    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        xf = x.float()
        xf = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return xf.to(x.dtype) * self.weight


class SwiGLU(nn.Module):
    """FFN(x) = W2( SiLU(W1 x) * (W3 x) )  - gated feed-forward (PaLM / LLaMA)."""

    def __init__(self, d_model, hidden, dropout):
        super().__init__()
        self.w1 = nn.Linear(d_model, hidden, bias=False)
        self.w3 = nn.Linear(d_model, hidden, bias=False)
        self.w2 = nn.Linear(hidden, d_model, bias=False)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.drop(self.w2(F.silu(self.w1(x)) * self.w3(x)))


class Attention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        assert cfg.d_model % cfg.n_heads == 0 and cfg.n_heads % cfg.n_kv_heads == 0
        self.h, self.kvh, self.dh = cfg.n_heads, cfg.n_kv_heads, cfg.d_model // cfg.n_heads
        self.q_proj = nn.Linear(cfg.d_model, self.h * self.dh, bias=False)
        self.k_proj = nn.Linear(cfg.d_model, self.kvh * self.dh, bias=False)
        self.v_proj = nn.Linear(cfg.d_model, self.kvh * self.dh, bias=False)
        self.o_proj = nn.Linear(self.h * self.dh, cfg.d_model, bias=False)
        self.dropout = cfg.dropout

    def forward(self, x, cos, sin, attn_mask=None, cache=None):
        """x (B,T,C); cos/sin for these T positions; attn_mask bool (B,1,T,S) True=attend or None=all;
        cache = (k, v) of earlier positions or None. Returns output and the updated (k, v)."""
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.h, self.dh).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.kvh, self.dh).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.kvh, self.dh).transpose(1, 2)
        q = apply_rope(q, cos.to(q.dtype), sin.to(q.dtype))
        k = apply_rope(k, cos.to(k.dtype), sin.to(k.dtype))
        if cache is not None:
            k = torch.cat([cache[0], k], dim=2)
            v = torch.cat([cache[1], v], dim=2)
        new_cache = (k, v)
        if self.kvh != self.h:                                   # GQA: share each K/V head
            k = k.repeat_interleave(self.h // self.kvh, dim=1)
            v = v.repeat_interleave(self.h // self.kvh, dim=1)
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask,
                                             dropout_p=self.dropout if self.training else 0.0)
        return self.o_proj(out.transpose(1, 2).reshape(B, T, -1)), new_cache


class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = RMSNorm(cfg.d_model)
        self.attn = Attention(cfg)
        self.norm2 = RMSNorm(cfg.d_model)
        self.ffn = SwiGLU(cfg.d_model, cfg.ffn_hidden, cfg.dropout)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x, cos, sin, attn_mask=None, cache=None):     # pre-norm residual block
        a, new_cache = self.attn(self.norm1(x), cos, sin, attn_mask, cache)
        x = x + self.drop(a)
        return x + self.ffn(self.norm2(x)), new_cache


class PrefixLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.norm = RMSNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight                      # weight tying
        cos, sin = build_rope_cache(cfg.max_seq_len, cfg.d_model // cfg.n_heads)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

    @staticmethod
    def prefix_mask(prefix_len, T):
        """prefix_len (B,) -> bool mask (B,1,T,T): i attends j  <=>  j <= i  or  j < p."""
        i = torch.arange(T, device=prefix_len.device)
        causal = i[None, :] <= i[:, None]                              # (T,T)  [i, j]: j <= i
        in_prefix = i[None, None, :] < prefix_len[:, None, None]       # (B,1,T): j < p
        return (causal[None] | in_prefix)[:, None]                     # (B,1,T,T)

    def _run(self, idx, attn_mask, start_pos=0, caches=None):
        T = idx.size(1)
        cos = self.rope_cos[start_pos:start_pos + T]
        sin = self.rope_sin[start_pos:start_pos + T]
        x = self.drop(self.tok_emb(idx))
        new_caches = []
        for li, block in enumerate(self.blocks):
            x, c = block(x, cos, sin, attn_mask, None if caches is None else caches[li])
            new_caches.append(c)
        return self.norm(x), new_caches

    def hidden(self, idx, prefix_len=None):
        """Final hidden states (B,T,d). Default = fully bidirectional (whole input is the prefix):
        the natural feature extractor for Phase 2+ (NER, classification, ...)."""
        B, T = idx.shape
        p = torch.full((B,), T if prefix_len is None else prefix_len, device=idx.device)
        return self._run(idx, self.prefix_mask(p, T))[0]

    def forward(self, idx, prefix_len=None):
        return self.head(self.hidden(idx, 0 if prefix_len is None else prefix_len))

    def compute_loss(self, x, y):
        """Next-token loss on positions i >= p-1 (positions that cannot see their target)."""
        B, T = x.shape
        if self.training:
            lo, hi = int(self.cfg.prefix_min * T), int(self.cfg.prefix_max * T)
            p = torch.randint(lo, hi + 1, (B,), device=x.device)
        else:
            p = torch.full((B,), int(self.cfg.eval_prefix * T), device=x.device)
        h, _ = self._run(x, self.prefix_mask(p, T))
        logits = self.head(h)
        pos = torch.arange(T, device=x.device)[None, :]
        target = y.masked_fill(pos < (p[:, None] - 1).clamp(min=0), -100)   # ignore leaky positions
        return F.cross_entropy(logits.float().reshape(-1, logits.size(-1)), target.reshape(-1),
                               ignore_index=-100)

    # ---------------- generation with a KV cache ----------------
    @torch.no_grad()
    def prefill(self, prompt):
        """Prompt (B,P) is the prefix: one bidirectional pass. Returns next-token logits (B,V) + cache."""
        h, caches = self._run(prompt, None)                       # None mask = every prompt token sees all
        return self.head(h[:, -1]), caches

    @torch.no_grad()
    def decode_step(self, token, caches, pos):
        """token (B,1) at absolute position pos; attends to all cached positions + itself."""
        h, caches = self._run(token, None, start_pos=pos, caches=caches)
        return self.head(h[:, -1]), caches

In [ ]:
%%writefile a3src/mla.py
# Architecture 4: decoder-only Transformer with Multi-head Latent Attention (MLA, DeepSeek-V2 style),
# RMSNorm, SwiGLU, decoupled RoPE and a compressed KV cache.
#
# Standard MHA caches K and V for every head: 2 * n_heads * d_head numbers per token per layer.
# MLA instead caches ONE small latent per token and re-creates per-head keys/values from it:
#     c_kv   = RMSNorm(W_dkv x)                       (kv_rank numbers, shared by all heads)
#     k_nope = W_uk c_kv ,  v = W_uv c_kv             (per head, "no position" part)
#     k_rope = RoPE(W_kr x)                           (rope_dim numbers, shared by all heads)
#     q      = [q_nope ; RoPE(q_rope)] = W_q x        (per head)
#     score  = (q_nope . k_nope + q_rope . k_rope) / sqrt(nope_dim + rope_dim)
# RoPE is "decoupled" (a separate small key) because a position rotation could not be moved
# through the up-projection W_uk. Cache per token per layer = kv_rank + rope_dim numbers.
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F
from rope import build_rope_cache, apply_rope
from prefixlm import RMSNorm, SwiGLU            # same building blocks as architecture 3


@dataclass
class MLAConfig:
    vocab_size: int
    d_model: int = 768
    n_layers: int = 12
    n_heads: int = 12
    kv_rank: int = 256            # size of the compressed key/value latent c_kv
    nope_dim: int = 64            # per-head query/key dimension without position
    rope_dim: int = 32            # per-head query / shared key dimension carrying RoPE
    v_dim: int = 64               # per-head value dimension
    ffn_hidden: int = 2048        # SwiGLU hidden size
    max_seq_len: int = 1024
    dropout: float = 0.0


class MLAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.h, self.nope, self.rope, self.vd = cfg.n_heads, cfg.nope_dim, cfg.rope_dim, cfg.v_dim
        self.q_proj = nn.Linear(cfg.d_model, self.h * (self.nope + self.rope), bias=False)
        self.kv_down = nn.Linear(cfg.d_model, cfg.kv_rank, bias=False)          # W_dkv
        self.kv_norm = RMSNorm(cfg.kv_rank)
        self.k_up = nn.Linear(cfg.kv_rank, self.h * self.nope, bias=False)      # W_uk
        self.v_up = nn.Linear(cfg.kv_rank, self.h * self.vd, bias=False)        # W_uv
        self.k_rope = nn.Linear(cfg.d_model, self.rope, bias=False)             # W_kr (shared by heads)
        self.o_proj = nn.Linear(self.h * self.vd, cfg.d_model, bias=False)
        self.dropout = cfg.dropout

    def forward(self, x, cos, sin, cache=None):
        """x (B,T,C); cos/sin for these T positions; cache = (c_kv, k_rope) of earlier tokens or None.
        Without a cache the attention is causal over x; with a cache the T new tokens see all cached
        tokens and (causally) each other. Returns output and the updated (compressed) cache."""
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.h, self.nope + self.rope).transpose(1, 2)
        q_nope, q_rope = q.split([self.nope, self.rope], dim=-1)
        q_rope = apply_rope(q_rope, cos.to(q.dtype), sin.to(q.dtype))
        c_kv = self.kv_norm(self.kv_down(x))                                    # (B,T,kv_rank)
        k_r = apply_rope(self.k_rope(x)[:, None], cos.to(q.dtype), sin.to(q.dtype))   # (B,1,T,rope)
        if cache is not None:
            c_kv = torch.cat([cache[0], c_kv], dim=1)
            k_r = torch.cat([cache[1], k_r], dim=2)
        new_cache = (c_kv, k_r)                                                 # <- all that is cached
        S = c_kv.size(1)
        k_nope = self.k_up(c_kv).view(B, S, self.h, self.nope).transpose(1, 2)
        v = self.v_up(c_kv).view(B, S, self.h, self.vd).transpose(1, 2)
        q = torch.cat([q_nope, q_rope], dim=-1)
        k = torch.cat([k_nope, k_r.expand(B, self.h, S, self.rope)], dim=-1)
        if cache is None:
            out = F.scaled_dot_product_attention(q, k, v, is_causal=True,
                                                 dropout_p=self.dropout if self.training else 0.0)
        else:                                                                   # new tokens at the end
            i = torch.arange(T, device=x.device)[:, None] + (S - T)
            mask = torch.arange(S, device=x.device)[None, :] <= i              # (T,S)
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
        return self.o_proj(out.transpose(1, 2).reshape(B, T, -1)), new_cache


class MLABlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = RMSNorm(cfg.d_model)
        self.attn = MLAttention(cfg)
        self.norm2 = RMSNorm(cfg.d_model)
        self.ffn = SwiGLU(cfg.d_model, cfg.ffn_hidden, cfg.dropout)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x, cos, sin, cache=None):
        a, new_cache = self.attn(self.norm1(x), cos, sin, cache)
        x = x + self.drop(a)
        return x + self.ffn(self.norm2(x)), new_cache


class MLALM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([MLABlock(cfg) for _ in range(cfg.n_layers)])
        self.norm = RMSNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight                                  # weight tying
        cos, sin = build_rope_cache(cfg.max_seq_len, cfg.rope_dim)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

    def _run(self, idx, start_pos=0, caches=None):
        T = idx.size(1)
        cos, sin = self.rope_cos[start_pos:start_pos + T], self.rope_sin[start_pos:start_pos + T]
        x = self.drop(self.tok_emb(idx))
        new = []
        for li, blk in enumerate(self.blocks):
            x, c = blk(x, cos, sin, None if caches is None else caches[li])
            new.append(c)
        return self.norm(x), new

    def hidden(self, idx):
        """Final (causal) hidden states (B,T,d) - feature extractor for later phases."""
        return self._run(idx)[0]

    def forward(self, idx):                     # plain next-token LM: loss on every position
        return self.head(self.hidden(idx))

    def kv_cache_floats_per_token(self):
        """Cached numbers per token per layer: MLA vs what standard MHA would store."""
        c = self.cfg
        return c.kv_rank + c.rope_dim, 2 * c.n_heads * c.v_dim

    # ---------------- generation with the compressed KV cache ----------------
    @torch.no_grad()
    def prefill(self, prompt):
        h, caches = self._run(prompt)
        return self.head(h[:, -1]), caches

    @torch.no_grad()
    def decode_step(self, token, caches, pos):
        h, caches = self._run(token, start_pos=pos, caches=caches)
        return self.head(h[:, -1]), caches

In [ ]:
%%writefile a3src/a1_tokenizer.py
"""Assignment-1 byte-level BPE tokenizer, retrained on the Assignment-3 corpus.

Byte-level BPE (A-1 Q5 spec): byte symbols so no <unk> is needed, reserved tokens
<pad> <unk> <s> </s> at ids 0..3, each merge takes the most frequent adjacent pair
(ties broken by the lexicographically smallest merged bytes), and merges never cross
a pre-token (word / whitespace) boundary so decode() reconstructs the text exactly.
"""
import re
from collections import Counter

RESERVED = ("<pad>", "<unk>", "<s>", "</s>")
BASE = len(RESERVED)
SPLIT = re.compile(r"\S+|\s+")


def merge_sequence(ids, pair, new_id):
    a, b = pair
    out = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == a and ids[i + 1] == b:
            out.append(new_id)
            i += 2
        else:
            out.append(ids[i])
            i += 1
    return tuple(out)


class BPETokenizer:
    def __init__(self):
        self.merges = {}
        self.ranks = {}
        self.itos = {}
        self.vocab_size = 0
        self.cache = {}

    def train(self, text, vocab_size):
        self.itos = {i: b"" for i in range(BASE)}
        for byte in range(256):
            self.itos[BASE + byte] = bytes((byte,))

        counts = Counter(SPLIT.findall(text))
        corpus = Counter()
        for word, freq in counts.items():
            ids = tuple(BASE + byte for byte in word.encode("utf-8"))
            corpus[ids] += freq

        n_merges = max(0, vocab_size - BASE - 256)
        next_id = BASE + 256
        while len(self.merges) < n_merges:
            pair_freq = Counter()
            for ids, freq in corpus.items():
                for j in range(len(ids) - 1):
                    pair_freq[(ids[j], ids[j + 1])] += freq
            if not pair_freq:
                break
            top = max(pair_freq.values())
            if top < 2:
                break
            best_pairs = [p for p, f in pair_freq.items() if f == top]
            pair = min(best_pairs, key=lambda p: self.itos[p[0]] + self.itos[p[1]])

            self.itos[next_id] = self.itos[pair[0]] + self.itos[pair[1]]
            self.merges[pair] = next_id
            self.ranks[pair] = len(self.ranks)

            new_corpus = Counter()
            for ids, freq in corpus.items():
                new_corpus[merge_sequence(ids, pair, next_id)] += freq
            corpus = new_corpus
            next_id += 1

        self.vocab_size = BASE + 256 + len(self.merges)

    def encode_chunk(self, raw_bytes):
        if raw_bytes in self.cache:
            return self.cache[raw_bytes]
        ids = [BASE + byte for byte in raw_bytes]
        while len(ids) >= 2:
            candidates = {}
            for j in range(len(ids) - 1):
                pair = (ids[j], ids[j + 1])
                if pair in self.ranks:
                    candidates[pair] = self.ranks[pair]
            if not candidates:
                break
            pair = min(candidates, key=candidates.get)
            ids = list(merge_sequence(ids, pair, self.merges[pair]))
        self.cache[raw_bytes] = ids
        return ids

    def encode(self, text):
        ids = []
        for chunk in SPLIT.findall(text):
            ids.extend(self.encode_chunk(chunk.encode("utf-8")))
        return ids

    def decode(self, ids):
        data = b""
        for i in ids:
            data += self.itos.get(i, b"")
        return data.decode("utf-8", errors="replace")

    def get_vocab_size(self):
        return self.vocab_size


def train_tokenizer(train_text, vocab_size):
    tokenizer = BPETokenizer()
    tokenizer.train(train_text, vocab_size)
    return tokenizer

In [ ]:
%%writefile a3src/space_bpe.py
# Space-attached byte-level BPE (GPT-2 style pre-tokenisation) built on your A1 tokenizer.
#
# Difference from A1: A1 splits text into  \S+ | \s+  so every space is its own token
# (~2.3 tokens per word). Here a single leading space is attached to the following word
# (" the", " Court"), so a word usually costs ~1.3 tokens and a 512-word prompt fits in the
# 1024-token context. Merge rule is unchanged (most frequent pair, ties -> lexicographically
# smallest merged bytes, never across a pre-token boundary), so decode() is still exact.
# train() is an incremental version of the A1 algorithm: it only re-counts pairs in words that
# contain the merged pair (plus a heap), which gives the same merges but runs minutes, not hours.
import heapq
import re
from collections import Counter, defaultdict
from a1_tokenizer import BPETokenizer, BASE, merge_sequence

# " ?\S+"      a word with at most one leading space
# "\s+(?!\S)"  a whitespace run, leaving its last space for the next word
# "\s+"        trailing whitespace at the very end
SPACE_SPLIT = re.compile(r" ?\S+|\s+(?!\S)|\s+")


class SpaceBPETokenizer(BPETokenizer):
    space_prefix = True          # tells the pipeline not to append a trailing space to prompts

    def train(self, text, vocab_size):
        self.itos = {i: b"" for i in range(BASE)}
        for byte in range(256):
            self.itos[BASE + byte] = bytes((byte,))
        counts = Counter(SPACE_SPLIT.findall(text))
        words = [tuple(BASE + b for b in w.encode("utf-8")) for w in counts]
        freqs = list(counts.values())

        pair_counts = defaultdict(int)          # pair -> total frequency
        where = defaultdict(set)                # pair -> indices of words containing it
        for wi, ids in enumerate(words):
            for pair in zip(ids, ids[1:]):
                pair_counts[pair] += freqs[wi]
                where[pair].add(wi)
        merged = lambda p: self.itos[p[0]] + self.itos[p[1]]
        heap = [(-c, merged(p), p) for p, c in pair_counts.items()]
        heapq.heapify(heap)

        n_merges = max(0, vocab_size - BASE - 256)
        next_id = BASE + 256
        while len(self.merges) < n_merges and heap:
            neg_c, _, pair = heapq.heappop(heap)
            c = pair_counts.get(pair, 0)
            if c != -neg_c or c <= 0:
                continue                         # stale heap entry
            if c < 2:
                break
            self.itos[next_id] = merged(pair)
            self.merges[pair] = next_id
            self.ranks[pair] = len(self.ranks)
            delta = defaultdict(int)
            for wi in where.pop(pair, ()):
                old = words[wi]
                new = merge_sequence(old, pair, next_id)
                if new == old:
                    continue
                f = freqs[wi]
                for p in zip(old, old[1:]):
                    delta[p] -= f
                for p in zip(new, new[1:]):
                    delta[p] += f
                    where[p].add(wi)
                words[wi] = new
            pair_counts.pop(pair, None)
            for p, d in delta.items():
                if d == 0 or p == pair:
                    continue
                pair_counts[p] += d
                if pair_counts[p] > 0:
                    heapq.heappush(heap, (-pair_counts[p], merged(p), p))
            next_id += 1
        self.vocab_size = BASE + 256 + len(self.merges)

    def encode(self, text):
        ids = []
        for chunk in SPACE_SPLIT.findall(text):
            ids.extend(self.encode_chunk(chunk.encode("utf-8")))
        return ids


def train_space_tokenizer(train_text, vocab_size):
    tok = SpaceBPETokenizer()
    tok.train(train_text, vocab_size)
    return tok

In [ ]:
# 4b · Imports, tokenizer save/load, AMP helper ---------------------------------
import sys, importlib
sys.path.insert(0, os.path.abspath("a3src"))
import rope, attention, model_arch, a1_tokenizer, encdec, prefixlm, mla, space_bpe
for _m in (rope, attention, model_arch, a1_tokenizer, encdec, prefixlm, mla, space_bpe): importlib.reload(_m)
from a1_tokenizer import BPETokenizer, train_tokenizer
from model_arch import LMConfig, DecoderLM
from encdec import EncDecConfig, EncDecLM
from prefixlm import PrefixLMConfig, PrefixLM
from mla import MLAConfig, MLALM
from space_bpe import SpaceBPETokenizer, train_space_tokenizer

def save_tokenizer(tok, path):
    cache = tok.cache; tok.cache = {}            # don't pickle the (large) encode cache
    try:
        with open(path, "wb") as f: pickle.dump(tok, f)
    finally:
        tok.cache = cache

def load_tokenizer(path):
    with open(path, "rb") as f: return pickle.load(f)

PAD, UNK, BOS, EOS = 0, 1, 2, 3                  # reserved ids in your A1 tokenizer
_use_cuda = (device == "cuda")
def amp_autocast():
    return torch.autocast(device_type=("cuda" if _use_cuda else "cpu"), enabled=_use_cuda)

if CONFIG["prebuilt_dir"]:
    # Kaggle datasets made from an output folder are often nested one level deeper -> search for meta.json
    found = [r for r, _, fs in os.walk(CONFIG["prebuilt_dir"]) if "meta.json" in fs]
    if not found:
        hits = [r for r, _, fs in os.walk("/kaggle/input") if "meta.json" in fs] if os.path.isdir("/kaggle/input") else []
        raise FileNotFoundError(
            f"prebuilt_dir={CONFIG['prebuilt_dir']!r} has no meta.json (dataset not attached, or wrong path).\n"
            f"Folders under /kaggle/input that DO contain meta.json: {hits or 'none - attach the dataset (Add Input)'}\n"
            "-> set prebuilt_dir to one of them.")
    DATA_DIR = found[0]
DATA_READY = os.path.exists(os.path.join(DATA_DIR, "meta.json"))
print("Imported the model/tokenizer code. Data dir:", DATA_DIR, "| prebuilt data found:", DATA_READY)

## 5 · Data: stream → document-level split → dev set
Skipped automatically if `DATA_DIR` already has `meta.json` (for example when `prebuilt_dir` points to a saved Kaggle dataset).
**Look at the printed samples**: are they real legal text? Do the dev prompts look like the official test prompts?

In [ ]:
# 5a · Discover files, download only the (small) test file ---------------------
from huggingface_hub import HfApi, HfFileSystem, hf_hub_download
api = HfApi()
repo_files  = api.list_repo_files(CONFIG["hf_repo"], repo_type="dataset")
phase_files = [f for f in repo_files if f.startswith(CONFIG["phase_dir"] + "/")]
print("Phase-1 files:", *phase_files, sep="\n  ")

train_files = {}
for src in CONFIG["sources"]:
    m = [f for f in phase_files if "/Train/" in f and src in f.lower() and f.endswith(".json")]
    if m: train_files[src] = m[0]
test_files = [f for f in phase_files if "/Test/" in f and f.endswith(".json")]
assert train_files and test_files, "Phase-1 train/test files not found."
test_path = hf_hub_download(CONFIG["hf_repo"], test_files[0], repo_type="dataset")
print("Train sources:", list(train_files), "| test file:", test_files[0])

In [ ]:
# 5b · Stream documents (no full download) and build the splits ------------------
import ijson
print("ijson backend:", ijson.backend)           # yajl2_c (C) is much faster than 'python'
fs = HfFileSystem()

def hf_open(fname):
    return fs.open(f"datasets/{CONFIG['hf_repo']}/{fname}", "rb", block_size=16 * 1024 * 1024)

def iter_texts(obj):
    """Yield strings from a record of unknown shape (dict/list/str)."""
    if isinstance(obj, str):
        if obj.strip(): yield obj
    elif isinstance(obj, dict):
        for k in ("text", "content", "body", "document", "judgment", "raw_text"):
            if isinstance(obj.get(k), str) and obj[k].strip():
                yield obj[k]; return
        for v in obj.values():
            yield from iter_texts(v)
    elif isinstance(obj, list):
        for v in obj:
            yield from iter_texts(v)

def toplevel_kind(fname):
    with hf_open(fname) as f:
        while True:
            c = f.read(1)
            if not c: return None
            if c.strip(): return c                # b"{" or b"["

def stream_docs(fname, max_chars, skip_chars=0):
    """Documents in file order. The first documents totalling >= skip_chars are skipped (same rule as the
    budget stop), so skip_chars = an earlier run's per-source budget gives exactly the docs it never read."""
    kind, got, skipped = toplevel_kind(fname), 0, 0
    with hf_open(fname) as f:
        gen = ijson.kvitems(f, "") if kind == b"{" else ijson.items(f, "item")
        for item in gen:
            val = item[1] if kind == b"{" else item
            for t in iter_texts(val):
                if skipped < skip_chars:
                    skipped += len(t); continue
                yield t; got += len(t)
                if got >= max_chars: return

def collect_docs(fname, budget, skip_chars=0, tries=3):
    """Streaming ~200 MB over the network can hiccup; retry the whole source (same docs every time)."""
    for attempt in range(1, tries + 1):
        try:
            return list(stream_docs(fname, budget, skip_chars))
        except Exception as e:
            print(f"  stream error on {fname} ({type(e).__name__}: {e}); retry {attempt}/{tries}")
            time.sleep(15)
    raise RuntimeError(f"could not stream {fname}")

def build_dev_set(docs, n, prompt_words=512, cont_words=128, seed=0):
    """Mimic the official task: a 512-word prompt and the next 128 words as reference."""
    rng = random.Random(seed)
    need = prompt_words + cont_words
    pools = [d.split() for d in docs]
    pools = [w for w in pools if len(w) >= need] or [sum(pools, [])]   # fallback: concatenate
    prompts, refs = {}, {}
    for i in range(n):
        w = rng.choice(pools)
        s = rng.randint(0, len(w) - need)
        prompts[str(i)] = {"text": " ".join(w[s:s + prompt_words])}
        refs[str(i)]    = {"text": " ".join(w[s + prompt_words:s + need])}
    return prompts, refs

if not DATA_READY:
    CHARS_PER_TOKEN_GUESS = 4.8 if CONFIG["tokenizer_type"] == "space" else 6   # generous; true value printed later
    os.makedirs(DATA_DIR, exist_ok=True)
    REUSE = CONFIG["tokenizer_from"]
    if REUSE:            # continued training: same tokenizer + the SAME val/dev sets -> comparable numbers
        _f = [r for r, _, fs in os.walk(REUSE) if "meta.json" in fs]
        assert _f, f"tokenizer_from={REUSE!r} has no meta.json (dataset attached?)"
        REUSE_DIR = _f[0]; old_meta = json.load(open(os.path.join(REUSE_DIR, "meta.json")))
        for fn in ("legal_bpe.pkl", "val.bin", "dev_prompts.json", "dev_refs.json"):
            shutil.copy(os.path.join(REUSE_DIR, fn), os.path.join(DATA_DIR, fn))
        print(f"Reusing tokenizer + val/dev sets from {REUSE_DIR} (fingerprint {old_meta.get('tok_hash')})")
    val_frac = 0.0 if REUSE else CONFIG["val_doc_frac"]
    skip = CONFIG["skip_chars_per_source"]
    char_budget = max(TRAIN_TOKENS * CHARS_PER_TOKEN_GUESS, 0 if REUSE else CONFIG["tok_train_chars"] * 1.2) / (1 - val_frac)
    wsum = sum(CONFIG["mix"][s] for s in train_files)
    budgets = {s: int(char_budget * CONFIG["mix"][s] / wsum) for s in train_files}
    docs_by_src, used = {}, {}
    for src, fname in train_files.items():
        t0 = time.time()
        docs_by_src[src] = collect_docs(fname, budgets[src], skip)
        used[src] = sum(map(len, docs_by_src[src]))
        print(f"{src:>14}: {len(docs_by_src[src]):>7,} docs | {used[src]:>13,} chars"
              + (f" (after skipping the first {skip:,})" if skip else "")
              + (" | SOURCE EXHAUSTED" if used[src] < budgets[src] else "") + f" | {time.time()-t0:.0f}s")
    # smaller sources can run out in a later slice: top up from the sources that still have text
    short = char_budget - sum(used.values())
    open_src = [s for s in docs_by_src if used[s] >= budgets[s]]
    if short > 0.02 * char_budget and open_src:
        for src in open_src:
            more = collect_docs(train_files[src], short / len(open_src), skip + used[src])
            docs_by_src[src] += more; used[src] += sum(map(len, more))
            print(f"  top-up {src}: +{len(more):,} docs, +{sum(map(len, more)):,} chars")
    rng = random.Random(CONFIG["seed"])
    train_pairs, val_by_src, source_stats = [], {}, {}
    for src, docs in docs_by_src.items():
        rng.shuffle(docs)
        n_val = 0 if REUSE else max(1, int(round(len(docs) * val_frac)))
        if not REUSE:
            val_by_src[src] = docs[:n_val]                  # held out PER SOURCE (stratified)
        train_pairs += [(src, d) for d in docs[n_val:]]
        source_stats[src] = dict(docs=len(docs), chars=used[src], val_docs=n_val, skipped_chars=skip)
    rng.shuffle(train_pairs)
    train_src = [s for s, _ in train_pairs]; train_docs = [d for _, d in train_pairs]; del train_pairs
    print(f"\nTrain docs: {len(train_docs):,} | Val docs: {sum(map(len, val_by_src.values())):,}"
          + (" (val/dev reused)" if REUSE else ""))
    print("\n--- SAMPLE train doc ---\n", train_docs[0][:500])
    if not REUSE:
        # dev set: equal number of examples from every source's held-out docs
        dev_prompts, dev_refs, per = {}, {}, max(1, CONFIG["n_dev_examples"] // len(val_by_src))
        for j, (src, vdocs) in enumerate(val_by_src.items()):
            p, r = build_dev_set(vdocs, per, seed=CONFIG["seed"] + j)
            for k in p:
                key = str(len(dev_prompts))
                dev_prompts[key] = {**p[k], "source": src}; dev_refs[key] = {**r[k], "source": src}
        json.dump(dev_prompts, open(os.path.join(DATA_DIR, "dev_prompts.json"), "w"), ensure_ascii=False)
        json.dump(dev_refs,    open(os.path.join(DATA_DIR, "dev_refs.json"), "w"), ensure_ascii=False)
    dev_prompts = json.load(open(os.path.join(DATA_DIR, "dev_prompts.json"), encoding="utf-8"))
    dev_refs = json.load(open(os.path.join(DATA_DIR, "dev_refs.json"), encoding="utf-8"))
    print("\n--- SAMPLE dev prompt (last 200 chars) ---\n", dev_prompts["0"]["text"][-200:])
    print("--- its reference (first 200 chars) ---\n", dev_refs["0"]["text"][:200])

In [ ]:
# 6 · Train (or load) your byte-level BPE tokenizer -----------------------------
tok_path = os.path.join(DATA_DIR, "legal_bpe.pkl")
if os.path.exists(tok_path):
    tokenizer = load_tokenizer(tok_path)
    print("Loaded tokenizer from", tok_path)
else:
    parts, n = [], 0                               # train-split docs only (no val leakage)
    for d in train_docs:
        parts.append(d); n += len(d) + 2
        if n >= CONFIG["tok_train_chars"]: break
    tok_text = "\n\n".join(parts)[:CONFIG["tok_train_chars"]]
    print(f"Training {CONFIG['tokenizer_type']} BPE (vocab={CONFIG['vocab_size']}) on {len(tok_text):,} chars ...")
    t0 = time.time()
    trainer = train_space_tokenizer if CONFIG["tokenizer_type"] == "space" else train_tokenizer
    tokenizer = trainer(tok_text, CONFIG["vocab_size"])
    save_tokenizer(tokenizer, tok_path)
    print(f"Done in {time.time()-t0:.0f}s -> {tok_path}")

VOCAB = tokenizer.get_vocab_size()
import hashlib                                    # fingerprint: detects a mismatched tokenizer/checkpoint
TOK_HASH = hashlib.md5(json.dumps(sorted([a, b, c] for (a, b), c in tokenizer.merges.items())).encode()).hexdigest()[:12]
print("Tokenizer fingerprint:", TOK_HASH)
assert VOCAB < 65535, "token ids are stored as uint16"
s = "In the Supreme Court of India, the appellant contended that Section 302 IPC"
print("Vocab size:", VOCAB, "| round-trip OK:", tokenizer.decode(tokenizer.encode(s)) == s)
print("Tokenizer:", type(tokenizer).__name__, "| pieces:", [tokenizer.decode([i]) for i in tokenizer.encode(s)])
print(f"Tokens per word on this sentence: {len(tokenizer.encode(s)) / len(s.split()):.2f}")

In [ ]:
# 6b · Encode the corpus in parallel -> train.bin / val.bin (uint16) -----------
from multiprocessing import Pool

def _encode_doc(doc):
    ids = tokenizer.encode(doc); ids.append(EOS)
    return np.asarray(ids, dtype=np.uint16)

def encode_docs(docs, max_tokens, workers=os.cpu_count() or 2):
    """Returns (token array, chars consumed, per-doc token counts of the docs used)."""
    out, n, chars, lens, t0 = np.empty(max_tokens, dtype=np.uint16), 0, 0, [], time.time()
    with Pool(workers) as pool:
        for i, arr in enumerate(pool.imap(_encode_doc, docs, chunksize=8)):
            k = min(len(arr), max_tokens - n)
            out[n:n + k] = arr[:k]; n += k; chars += len(docs[i]); lens.append(k)
            if i % 2000 == 0:
                print(f"  {i:,} docs -> {n:,} tokens ({time.time()-t0:.0f}s)")
            if n >= max_tokens: break
    return out[:n], chars, lens

if not DATA_READY:
    # validation: equal token budget per source, stored as contiguous blocks (ranges saved in meta)
    val_parts, val_ranges, pos = [], {}, 0
    if REUSE:
        val_arr = np.fromfile(os.path.join(DATA_DIR, "val.bin"), dtype=np.uint16)
        val_ranges = old_meta.get("val_ranges") or {}
    else:
        for src, vdocs in val_by_src.items():
            arr, _, _ = encode_docs(vdocs, CONFIG["val_tokens"] // len(val_by_src))
            val_parts.append(arr); val_ranges[src] = [pos, pos + len(arr)]; pos += len(arr)
        val_arr = np.concatenate(val_parts)
    train_arr, train_chars, lens = encode_docs(train_docs, TRAIN_TOKENS)
    train_tokens_by_source = {s: 0 for s in docs_by_src}
    for s, k in zip(train_src, lens): train_tokens_by_source[s] += k
    train_arr.tofile(os.path.join(DATA_DIR, "train.bin"))
    val_arr.tofile(os.path.join(DATA_DIR, "val.bin"))
    meta = dict(vocab_size=VOCAB, n_train=int(len(train_arr)), n_val=int(len(val_arr)),
                chars_per_token=round(train_chars / max(1, len(train_arr)), 3),
                train_tokens_by_source=train_tokens_by_source, val_ranges=val_ranges,
                mix=CONFIG["mix"], source_stats=source_stats, val_doc_frac=CONFIG["val_doc_frac"],
                tok_train_chars=CONFIG["tok_train_chars"], seed=CONFIG["seed"], tok_hash=TOK_HASH,
                train_tokens_budget=TRAIN_TOKENS)
    meta.update(skip_chars_per_source=skip, eval_sets_reused_from=REUSE_DIR if REUSE else None)
    json.dump(meta, open(os.path.join(DATA_DIR, "meta.json"), "w"), indent=2)
    del train_docs, train_src, train_arr, val_arr, val_parts, val_by_src, docs_by_src   # free RAM
    DATA_READY = True

meta = json.load(open(os.path.join(DATA_DIR, "meta.json")))
train_data = np.memmap(os.path.join(DATA_DIR, "train.bin"), dtype=np.uint16, mode="r")
val_data   = np.memmap(os.path.join(DATA_DIR, "val.bin"),   dtype=np.uint16, mode="r")
dev_prompts = json.load(open(os.path.join(DATA_DIR, "dev_prompts.json"), encoding="utf-8"))
dev_refs    = json.load(open(os.path.join(DATA_DIR, "dev_refs.json"),    encoding="utf-8"))
epochs = CONFIG["max_steps"] * TOKENS_PER_STEP / len(train_data)
print(f"Train tokens: {len(train_data):,} | Val tokens: {len(val_data):,} | "
      f"chars/token: {meta['chars_per_token']} | dev examples: {len(dev_prompts)}")
print(f"This run will make {epochs:.2f} passes over the training tokens.")
if epochs > 1.05: print("NOTE: >1 pass means repeated data (this architecture uses more steps/tokens than arch-1).")
assert meta.get("tok_hash") in (None, TOK_HASH), "tokenizer does not match the one this data was encoded with"
for src, n in meta.get("train_tokens_by_source", {}).items():
    r = meta.get("val_ranges", {}).get(src, [0, 0])
    print(f"  {src:>14}: train {n:>12,} tokens ({n / max(1, len(train_data)):.1%}) | val {r[1]-r[0]:>10,} tokens")

## 7 · Model: build, initialise, sanity-check
The **initial loss must be ≈ ln(V)**. A randomly initialised model should be about as good as a uniform guess. Draft-1 started far above ln(V) (see FIX-1).

In [ ]:
# 7 · Build your DecoderLM, init weights, check the parameter budget -----------
if CONFIG["arch"] == "encdec":                    # architecture 2
    cfg = EncDecConfig(vocab_size=VOCAB, d_model=CONFIG["d_model"], n_heads=CONFIG["n_heads"],
                       d_ff=CONFIG["d_ff"], enc_layers=CONFIG["enc_layers"], dec_layers=CONFIG["dec_layers"],
                       enc_len=CONFIG["enc_len"], dec_len=CONFIG["dec_len"], dropout=CONFIG["dropout"],
                       bos_id=BOS)
    model = EncDecLM(cfg)
elif CONFIG["arch"] == "prefixlm":                # architecture 3
    cfg = PrefixLMConfig(vocab_size=VOCAB, d_model=CONFIG["d_model"], n_layers=CONFIG["n_layers"],
                         n_heads=CONFIG["n_heads"], n_kv_heads=CONFIG["n_kv_heads"],
                         ffn_hidden=CONFIG["ffn_hidden"], max_seq_len=CONFIG["max_seq_len"],
                         dropout=CONFIG["dropout"], prefix_min=CONFIG["prefix_min"],
                         prefix_max=CONFIG["prefix_max"], eval_prefix=CONFIG["eval_prefix"])
    model = PrefixLM(cfg)
elif CONFIG["arch"] == "mla":                     # architecture 4
    cfg = MLAConfig(vocab_size=VOCAB, d_model=CONFIG["d_model"], n_layers=CONFIG["n_layers"],
                    n_heads=CONFIG["n_heads"], kv_rank=CONFIG["mla_kv_rank"], nope_dim=CONFIG["mla_nope_dim"],
                    rope_dim=CONFIG["mla_rope_dim"], v_dim=CONFIG["mla_v_dim"], ffn_hidden=CONFIG["ffn_hidden"],
                    max_seq_len=CONFIG["max_seq_len"], dropout=CONFIG["dropout"])
    model = MLALM(cfg)
    mla_c, mha_c = model.kv_cache_floats_per_token()
    print(f"KV cache per token per layer: MLA {mla_c} numbers vs standard MHA {mha_c} -> {mha_c/mla_c:.1f}x smaller")
else:                                             # architecture 1
    cfg = LMConfig(
        vocab_size=VOCAB, d_model=CONFIG["d_model"], n_layers=CONFIG["n_layers"],
        n_heads=CONFIG["n_heads"], n_kv_heads=CONFIG["n_kv_heads"], d_ff=CONFIG["d_ff"],
        max_seq_len=CONFIG["max_seq_len"], dropout=CONFIG["dropout"], pos_emb_type=CONFIG["pos_emb_type"],
        norm_first=CONFIG["norm_first"], scale_emb=CONFIG["scale_emb"],
    )
    model = DecoderLM(cfg)
print("Architecture:", type(model).__name__, "|", cfg)

def init_weights(m):                              # [FIX-1] GPT-2 style initialisation
    if isinstance(m, nn.Linear):
        nn.init.normal_(m.weight, mean=0.0, std=0.02)
        if m.bias is not None: nn.init.zeros_(m.bias)
    elif isinstance(m, nn.Embedding):
        nn.init.normal_(m.weight, mean=0.0, std=0.02)
model.apply(init_weights)
model.to(device)

n_params = sum(p.numel() for p in model.parameters())      # tied weights are counted once
n_emb = next(m for m in model.modules() if isinstance(m, nn.Embedding)).weight.numel()
print(f"Parameters: {n_params:,} (non-embedding {n_params - n_emb:,})")
if not SMOKE_TEST:
    assert 50_000_000 <= n_params <= 300_000_000, "Phase-1 model must be 50M-300M params."
    print("Within the 50M-300M Phase-1 budget.")

T, BS, ACC = cfg.max_seq_len, CONFIG["batch_size"], CONFIG["grad_accum"]
assert len(val_data) > T + 2 and len(train_data) > T + 2, "not enough tokens for one window"
g_batch = torch.Generator().manual_seed(CONFIG["seed"])

# Training data = prebuilt_dir's train.bin, plus any extra_train_dirs (same tokenizer), sampled by train_mix.
TRAIN_SETS, TRAIN_W = [train_data], [len(train_data)]
for d in CONFIG["extra_train_dirs"]:
    _f = [r for r, _, fs in os.walk(d) if "meta.json" in fs]
    assert _f, f"extra_train_dirs: no meta.json under {d!r} (dataset attached?)"
    _m = json.load(open(os.path.join(_f[0], "meta.json")))
    assert _m.get("tok_hash") in (None, TOK_HASH), f"{d}: different tokenizer ({_m.get('tok_hash')} vs {TOK_HASH})"
    TRAIN_SETS.append(np.memmap(os.path.join(_f[0], "train.bin"), dtype=np.uint16, mode="r"))
    TRAIN_W.append(len(TRAIN_SETS[-1]))
if CONFIG["train_mix"]:
    assert len(CONFIG["train_mix"]) == len(TRAIN_SETS), "train_mix needs one weight per training set"
    TRAIN_W = list(CONFIG["train_mix"])
TRAIN_P = torch.tensor(TRAIN_W, dtype=torch.float64) / sum(TRAIN_W)
if len(TRAIN_SETS) > 1:
    _tot = sum(len(a) for a in TRAIN_SETS)
    print(f"Training on {len(TRAIN_SETS)} datasets ({_tot:,} tokens), sampling weights {[round(float(p), 3) for p in TRAIN_P]}:")
    for i, a in enumerate(TRAIN_SETS):
        _seen = CONFIG["max_steps"] * TOKENS_PER_STEP * float(TRAIN_P[i])
        print(f"   set {i}: {len(a):>13,} tokens -> ~{_seen / len(a):.2f} passes over it in this run")

def get_batch(data, bs, starts=None):
    if starts is None and data is train_data and len(TRAIN_SETS) > 1:
        which = torch.multinomial(TRAIN_P, bs, replacement=True, generator=g_batch).tolist()
        rows = []
        for w in which:
            arr = TRAIN_SETS[w]
            s = int(torch.randint(0, len(arr) - T - 1, (1,), generator=g_batch))
            rows.append((arr, s))
        x = torch.from_numpy(np.stack([a[s:s + T] for a, s in rows]).astype(np.int64))
        y = torch.from_numpy(np.stack([a[s + 1:s + 1 + T] for a, s in rows]).astype(np.int64))
        return x.to(device, non_blocking=True), y.to(device, non_blocking=True)
    if starts is None:
        starts = torch.randint(0, len(data) - T - 1, (bs,), generator=g_batch).tolist()
    x = torch.from_numpy(np.stack([data[s:s + T] for s in starts]).astype(np.int64))
    y = torch.from_numpy(np.stack([data[s + 1:s + 1 + T] for s in starts]).astype(np.int64))
    return x.to(device, non_blocking=True), y.to(device, non_blocking=True)

# [FIX-8] fixed, evenly spaced validation windows -> comparable across runs/architectures.
# Equal number of windows inside each source's block of val.bin (sources are weighted equally).
n_win = CONFIG["eval_batches"] * BS
ranges = meta.get("val_ranges") or {"all": [0, len(val_data)]}
ok = {s: r for s, r in ranges.items() if r[1] - r[0] > T + 2} or {"all": [0, len(val_data)]}
if len(ok) < len(ranges): print("WARNING: some sources have too few val tokens:", set(ranges) - set(ok))
VAL_STARTS_BY_SRC = {s: np.linspace(a, b - T - 2, max(1, n_win // len(ok))).astype(int).tolist()
                     for s, (a, b) in ok.items()}
VAL_STARTS = sum(VAL_STARTS_BY_SRC.values(), [])

def lm_loss(net, x, y):
    """Models with their own objective (enc-dec: loss on continuation tokens) define compute_loss."""
    if hasattr(net, "compute_loss"):
        return net.compute_loss(x, y)
    return F.cross_entropy(net(x).float().reshape(-1, VOCAB), y.reshape(-1))

@torch.no_grad()
def evaluate(starts=None):
    starts = starts or VAL_STARTS
    was_training = model.training; model.eval(); total = 0.0
    for i in range(0, len(starts), BS):
        x, y = get_batch(val_data, BS, starts[i:i + BS])
        with amp_autocast():
            total += lm_loss(model, x, y).item() * x.size(0)
    model.train(was_training)
    l = total / len(starts)
    return l, math.exp(min(20, l))

def evaluate_by_source():
    return {s: dict(zip(("val_loss", "val_ppl"), evaluate(st))) for s, st in VAL_STARTS_BY_SRC.items()}

init_loss, _ = evaluate()
print(f"Initial val loss {init_loss:.3f} vs ln(V) = {math.log(VOCAB):.3f}")
if init_loss > math.log(VOCAB) + 1.0:
    print("WARNING: initial loss is far above ln(V) -> check initialisation.")

In [ ]:
# 8 · Train from scratch (resumable) --------------------------------------------
try:
    scaler = torch.amp.GradScaler("cuda", enabled=_use_cuda)
except Exception:
    scaler = torch.cuda.amp.GradScaler(enabled=_use_cuda)

def lr_at(step):
    w, total, peak = CONFIG["warmup_steps"], CONFIG["max_steps"], CONFIG["lr"]
    if step < w: return peak * step / max(1, w)
    p = min(1.0, (step - w) / max(1, total - w))
    lo = peak * CONFIG["min_lr_ratio"]
    return lo + 0.5 * (peak - lo) * (1 + math.cos(math.pi * p))

# no weight decay on biases / LayerNorm (1-D params)
decay    = [p for _, p in model.named_parameters() if p.dim() >= 2]
no_decay = [p for _, p in model.named_parameters() if p.dim() < 2]
opt = torch.optim.AdamW([{"params": decay, "weight_decay": CONFIG["weight_decay"]},
                         {"params": no_decay, "weight_decay": 0.0}],
                        lr=CONFIG["lr"], betas=(0.9, 0.95))

CK_LAST = os.path.join(RUN_DIR, "ckpt_last.pt")
CK_BEST = os.path.join(RUN_DIR, "ckpt_best.pt")
HIST    = os.path.join(RUN_DIR, "history.json")

def atomic_save(obj, path):
    torch.save(obj, path + ".tmp"); os.replace(path + ".tmp", path)

def save_last(step, best, history):
    atomic_save({"model": model.state_dict(), "opt": opt.state_dict(), "scaler": scaler.state_dict(),
                 "step": step, "best": best, "history": history, "cfg": asdict(cfg), "config": CONFIG, "tok_hash": TOK_HASH,
                 "torch_rng": torch.get_rng_state(), "g_batch": g_batch.get_state(),
                 "cuda_rng": torch.cuda.get_rng_state_all() if _use_cuda else None, "ema": ema}, CK_LAST)

def save_best(step, vl, weights="raw"):
    atomic_save({"model": model.state_dict(), "cfg": asdict(cfg), "config": CONFIG, "tok_hash": TOK_HASH,
                 "step": step, "val_loss": vl, "weights": weights}, CK_BEST)

# Resume: a ckpt_last.pt already in this run folder (same session) is always the newest;
# otherwise use CONFIG["resume_from"] (e.g. a checkpoint attached from /kaggle/input).
start_step, best, history = 1, float("inf"), []
resume = CK_LAST if os.path.exists(CK_LAST) else CONFIG["resume_from"]
if resume:
    assert os.path.exists(resume), f"resume_from not found: {resume}"
    st = torch.load(resume, map_location=device, weights_only=False)
    assert st.get("tok_hash") in (None, TOK_HASH), (
        f"Checkpoint was trained with a different tokenizer ({st['tok_hash']} vs {TOK_HASH}): wrong data attached?")
    ck_arch = (st.get("config") or {}).get("arch", "decoder")
    assert ck_arch == CONFIG["arch"], (
        f"{resume} is an arch={ck_arch!r} checkpoint but this notebook builds arch={CONFIG['arch']!r}: "
        f"use the notebook for {ck_arch!r} (or set CONFIG arch='{ck_arch}' with the same sizes it was trained with).")
    model.load_state_dict(st["model"])
    if "opt" not in st:
        # a ckpt_best.pt (weights only): no training, go straight to evaluation + generation
        if not os.path.exists(CK_BEST): shutil.copy(resume, CK_BEST)
        start_step, best = CONFIG["max_steps"] + 1, st.get("val_loss", float("inf"))
        print(f"Loaded trained weights from {resume} (step {st.get('step')}) -> skipping training.")
    else:
        opt.load_state_dict(st["opt"]); scaler.load_state_dict(st["scaler"])
        # RNG states must be CPU ByteTensors (map_location=cuda moved them to the GPU)
        torch.set_rng_state(st["torch_rng"].cpu()); g_batch.set_state(st["g_batch"].cpu())
        if _use_cuda and st.get("cuda_rng") is not None:
            torch.cuda.set_rng_state_all([r.cpu() for r in st["cuda_rng"]])
        start_step, best, history = st["step"] + 1, st["best"], st["history"]
        src_best = os.path.join(os.path.dirname(os.path.abspath(resume)), "ckpt_best.pt")
        if not os.path.exists(CK_BEST) and os.path.exists(src_best):
            shutil.copy(src_best, CK_BEST)                  # bring the matching best checkpoint along
        if not os.path.exists(CK_BEST): best = float("inf")
        print(f"Resumed from {resume} at step {start_step} (best val so far {best:.3f})")

_resumed_ema = st.get("ema") if resume else None

# ---- continue from another run's weights with a NEW schedule (only when not resuming this run)
if not resume and CONFIG["init_from"]:
    st0 = torch.load(CONFIG["init_from"], map_location=device, weights_only=False)
    assert st0.get("tok_hash") in (None, TOK_HASH), "init_from checkpoint was trained with a different tokenizer"
    _a = (st0.get("config") or {}).get("arch", "decoder")
    assert _a == CONFIG["arch"], f"init_from is an arch={_a!r} checkpoint but this notebook builds {CONFIG['arch']!r}"
    model.load_state_dict(st0["model"])
    _opt_ok = CONFIG["init_load_optimizer"] and "opt" in st0
    if _opt_ok:
        opt.load_state_dict(st0["opt"]); scaler.load_state_dict(st0["scaler"])
    v0, _ = evaluate()
    print(f"Initialised from {CONFIG['init_from']} (its step {st0.get('step')}, optimizer "
          f"{'restored' if _opt_ok else 'fresh'}): val loss {v0:.3f} -> new schedule of {CONFIG['max_steps']} steps")
    del st0

# ---- EMA of the weights (evaluated alongside the raw weights; the better one is saved as best)
EMA_D = CONFIG["ema_decay"]
ema = None
if EMA_D > 0:
    ema = {k: v.detach().clone().float() for k, v in model.state_dict().items() if v.dtype.is_floating_point}
    if _resumed_ema is not None:
        for k in ema: ema[k].copy_(_resumed_ema[k])
    print(f"EMA of weights on (decay {EMA_D})")

def _swap_weights(new):
    """Copy `new` into the model in place; return the previous values (to swap back)."""
    cur = model.state_dict(); old = {k: cur[k].detach().clone() for k in new}
    with torch.no_grad():
        for k, v in new.items(): cur[k].copy_(v.to(cur[k].dtype))
    return old

# ---- data-parallel training on all GPUs (same tokens per optimizer step, ~2x the throughput)
_ng = CONFIG["train_gpus"] or (torch.cuda.device_count() if _use_cuda else 1)
DP, MB, ACC_RUN = None, BS, ACC
if _use_cuda and _ng > 1 and ACC % _ng == 0:
    class _LossWrap(nn.Module):
        def __init__(self, m):
            super().__init__(); self.m = m
        def forward(self, x, y):                  # runs once per GPU replica (in its own thread)
            with amp_autocast():
                return lm_loss(self.m, x, y).float().reshape(1)
    DP = nn.DataParallel(_LossWrap(model), device_ids=list(range(_ng)))
    MB, ACC_RUN = BS * _ng, ACC // _ng
    print(f"Training on {_ng} GPUs (DataParallel): micro-batch {MB} x {ACC_RUN} accumulation "
          f"= {MB * ACC_RUN * T:,} tokens/step (unchanged)")
elif _use_cuda and _ng > 1:
    print(f"Single-GPU training (grad_accum={ACC} is not divisible by {_ng} GPUs)")

train_deadline_h = CONFIG["session_limit_hours"] - CONFIG["reserve_hours_after_training"]
print(f"Session time used so far: {session_hours():.2f} h | training will stop at {train_deadline_h:.1f} h")

LOG_EVERY = max(1, CONFIG["eval_every"] // 5)
model.train(); t0 = t_log = time.time(); run_loss = torch.zeros((), device=device); n_run = 0
step = start_step - 1
for step in range(start_step, CONFIG["max_steps"] + 1):
    lr = lr_at(step)
    for g in opt.param_groups: g["lr"] = lr
    opt.zero_grad(set_to_none=True)
    for _ in range(ACC_RUN):
        x, y = get_batch(train_data, MB)
        if DP is not None:
            loss = DP(x, y).mean() / ACC_RUN
        else:
            with amp_autocast():
                loss = lm_loss(model, x, y) / ACC_RUN
        scaler.scale(loss).backward()
        run_loss += loss.detach()
    scaler.unscale_(opt)
    gnorm = torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["grad_clip"])
    scaler.step(opt); scaler.update(); n_run += 1
    if ema is not None:
        with torch.no_grad():
            _sd = model.state_dict()
            for k, e in ema.items(): e.mul_(EMA_D).add_(_sd[k].float(), alpha=1 - EMA_D)

    if step % LOG_EVERY == 0:
        dt = time.time() - t_log; t_log = time.time()
        tl = (run_loss / n_run).item(); run_loss.zero_(); n_run = 0
        eta = (CONFIG["max_steps"] - step) * dt / LOG_EVERY / 60
        print(f"step {step:>5} | lr {lr:.2e} | train {tl:.3f} | gnorm {float(gnorm):.2f} | "
              f"{LOG_EVERY * TOKENS_PER_STEP / dt:,.0f} tok/s | ETA {eta:.0f} min")
    if step % CONFIG["eval_every"] == 0 or step == CONFIG["max_steps"]:
        vl, vppl = evaluate(); vl_raw, vl_ema, src_w, raw_backup = vl, None, "raw", None
        if ema is not None:
            raw_backup = _swap_weights(ema)            # model now holds the EMA weights
            vl_ema, vppl_e = evaluate()
            if vl_ema < vl:
                vl, vppl, src_w = vl_ema, vppl_e, "ema"
            else:
                _swap_weights(raw_backup); raw_backup = None
        history.append(dict(step=step, tokens=step * TOKENS_PER_STEP, lr=lr, val_loss=vl, val_ppl=vppl,
                            val_loss_raw=vl_raw, val_loss_ema=vl_ema,
                            train_loss=tl if step % LOG_EVERY == 0 else None,
                            elapsed_min=(time.time() - t0) / 60))
        json.dump(history, open(HIST, "w"), indent=1)
        msg = f"  >> eval step {step}: val loss {vl:.3f} | val ppl {vppl:.1f}"
        if ema is not None:
            msg += f" (raw {vl_raw:.3f} / EMA {vl_ema:.3f} -> {src_w})"
        if vl < best:
            best = vl; save_best(step, vl, src_w)      # the model holds the chosen weights right now
            msg += "  (new best, saved)"
        if raw_backup is not None:
            _swap_weights(raw_backup)                  # back to the raw weights for training
        print(msg)
    if step % CONFIG["ckpt_every"] == 0:
        save_last(step, best, history)
    if session_hours() > train_deadline_h:
        print(f"Session time {session_hours():.1f} h reached at step {step}: stopping training so that "
              f"generation + zip still finish before Kaggle's 12 h limit (resume later with resume_from).")
        break
if step >= start_step: save_last(step, best, history)   # only if this session actually trained
if not os.path.exists(CK_BEST):                     # e.g. resumed at the final step with no best ckpt
    vl, _ = evaluate(); best = vl; save_best(step, vl)
TRAIN_MINUTES = (time.time() - t0) / 60
print(f"Training done. Best val loss {best:.3f} (ppl {math.exp(min(20, best)):.1f}) in {TRAIN_MINUTES:.0f} min.")

In [ ]:
# 8b · Loss curves (saved for the report) ----------------------------------------
import matplotlib.pyplot as plt
h = [r for r in history if r["val_loss"] is not None]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot([r["step"] for r in h], [r["val_loss"] for r in h], "o-", label="val")
tr = [r for r in h if r["train_loss"] is not None]
ax[0].plot([r["step"] for r in tr], [r["train_loss"] for r in tr], "s--", label="train (recent avg)")
ax[0].set_xlabel("step"); ax[0].set_ylabel("loss"); ax[0].legend(); ax[0].set_title(RUN_NAME)
ax[1].plot([r["tokens"] / 1e6 for r in h], [r["val_ppl"] for r in h], "o-")
ax[1].set_xlabel("tokens seen (M)"); ax[1].set_ylabel("val perplexity"); ax[1].set_yscale("log")
plt.tight_layout(); plt.savefig(os.path.join(RUN_DIR, "loss_curves.png"), dpi=120); plt.show()

In [ ]:
# 8c · Load the best checkpoint for generation ------------------------------------
state = torch.load(CK_BEST, map_location=device, weights_only=False)
model.load_state_dict(state["model"]); model.eval()
print(f"Loaded best checkpoint (step {state['step']}, val loss {state['val_loss']:.3f}).")

In [ ]:
# 9 · Batched generation: temperature / top-k / top-p / repetition penalty -------
SPECIAL_IDS = [PAD, UNK, BOS, EOS]

def first_n_words(text, n):
    return " ".join(text.split()[:n])

def sample_next(logits, window, temperature, top_k, top_p, repetition_penalty, gen):
    """logits (B,V) float -> next token ids (B,1)."""
    logits[:, SPECIAL_IDS] = float("-inf")                     # [FIX-4] never stop / emit specials
    if repetition_penalty != 1.0:                              # CTRL-style penalty on tokens in the window
        sc = logits.gather(1, window)
        sc = torch.where(sc > 0, sc / repetition_penalty, sc * repetition_penalty)
        logits.scatter_(1, window, sc)
    if temperature <= 0:                                       # greedy
        return logits.argmax(-1, keepdim=True)
    logits = logits / temperature
    if top_k:
        v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
        logits[logits < v[:, [-1]]] = float("-inf")
    probs = F.softmax(logits, dim=-1)
    if top_p and top_p < 1.0:
        sp, si = torch.sort(probs, descending=True)
        keep = (torch.cumsum(sp, dim=-1) - sp) <= top_p
        sp = sp * keep; sp = sp / sp.sum(dim=-1, keepdim=True)
        return si.gather(-1, torch.multinomial(sp, 1, generator=gen))
    return torch.multinomial(probs, 1, generator=gen)

@torch.no_grad()
def generate_batch(enc_prompts, gen_words, max_new_tokens, temperature, top_k, top_p,
                   repetition_penalty=1.0, ctx_tokens=None, seed=0, net=None, dev=None):
    """enc_prompts: list of token-id lists. All prompts are cut to the same length P (their last P
    tokens), so the batch needs no padding mask. Generates until every row has > gen_words words.
    net/dev: which model copy / device to use (for multi-GPU generation); default = model/device."""
    net = net or GEN_MODEL; dev = dev or next(net.parameters()).device
    encdec_model = hasattr(net, "encode")
    kv_model = hasattr(net, "prefill")                         # prefix-LM with a KV cache
    if kv_model:
        Tm = net.cfg.max_seq_len
        cap = ctx_tokens or max(16, Tm - CONFIG["gen_reserve_tokens"])
    elif encdec_model:
        Tm = net.cfg.dec_len; cap = net.cfg.enc_len
    else:
        Tm = cfg.max_seq_len; cap = ctx_tokens or Tm - 1
    P = min(min(len(e) for e in enc_prompts), cap)
    ctx = torch.tensor([e[-P:] for e in enc_prompts], dtype=torch.long, device=dev)
    gen = torch.Generator(device=dev); gen.manual_seed(seed)
    if kv_model:
        cache_start = 0                                        # ctx index where the cached sequence begins
        with amp_autocast():
            logits, caches = net.prefill(ctx)                  # prompt = bidirectional prefix, one pass
    if encdec_model:
        E, D = net.cfg.enc_len, net.cfg.dec_len
        bos = torch.full((ctx.size(0), 1), net.cfg.bos_id, dtype=torch.long, device=dev)
        cut = ctx.size(1)                                      # tokens before `cut` feed the encoder
        with amp_autocast():
            memory = net.encode(ctx[:, -E:])                   # prompt -> encoder, once
    for step in range(max_new_tokens):
        if kv_model:
            window, lg = ctx[:, -Tm:], logits.float()
        elif encdec_model:
            if ctx.size(1) - cut + 1 > D:
                # decoder window full: re-encode everything generated so far and restart the decoder
                # from <s> + the last D/2 tokens -> exactly the prompt/continuation split seen in training
                cut = ctx.size(1) - D // 2
                with amp_autocast():
                    memory = net.encode(ctx[:, :cut][:, -E:])
            window = torch.cat([bos, ctx[:, cut:]], dim=1)
            with amp_autocast():
                lg = net.decode(memory, window)[:, -1, :].float()
        else:
            window = ctx[:, -Tm:]                              # sliding window beyond max_seq_len
            with amp_autocast():
                lg = net(window)[:, -1, :].float()
        nxt = sample_next(lg, window, temperature, top_k, top_p, repetition_penalty, gen)
        ctx = torch.cat([ctx, nxt], dim=1)
        if kv_model:
            pos = ctx.size(1) - 1 - cache_start                # position of nxt in the cached sequence
            with amp_autocast():
                if pos >= Tm:
                    # cache full: re-prefill with the recent text as a new (bidirectional) prefix
                    cache_start = ctx.size(1) - max(16, Tm - CONFIG["gen_reserve_tokens"])
                    logits, caches = net.prefill(ctx[:, cache_start:])
                else:
                    logits, caches = net.decode_step(nxt, caches, pos)
        if (step + 1) % 16 == 0:                               # [FIX-4] stop once all rows have >128 words
            texts = [tokenizer.decode(r) for r in ctx[:, P:].tolist()]
            if all(len(t.split()) > gen_words for t in texts): break
    return [first_n_words(tokenizer.decode(r), gen_words) for r in ctx[:, P:].tolist()]

import copy, threading, queue

def gen_devices():
    """All GPUs to generate on (Kaggle 'GPU T4 x2' -> 2). CONFIG['gen_gpus'] can limit the number."""
    if not _use_cuda: return [torch.device(device)]
    n = torch.cuda.device_count()
    n = min(n, CONFIG["gen_gpus"]) if CONFIG.get("gen_gpus") else n
    return [torch.device(f"cuda:{i}") for i in range(max(1, n))]

def generate_all(prompt_dict, dec, bs=None, log_every=10, log_secs=300):
    """prompt_dict: {id: {"text": prompt}} -> {id: {"text": 128-word continuation}}.
    Batches are shared out across all GPUs through a work queue: one model copy + one thread per GPU
    (PyTorch releases the GIL inside CUDA ops, so the GPUs really run in parallel).
    Each batch keeps its own seed, so the output is the same whatever the number of GPUs."""
    bs = bs or CONFIG["gen_batch_size"]
    ids = list(prompt_dict)
    # A1 tokenizer: append a space so the continuation starts a NEW word. The space-attached tokenizer
    # puts the space inside the next word token (" the"), so there nothing is appended.
    suffix = "" if getattr(tokenizer, "space_prefix", False) else " "
    enc = {k: tokenizer.encode(prompt_dict[k]["text"].rstrip() + suffix) for k in ids}
    order = sorted(ids, key=lambda k: len(enc[k]))             # similar lengths per batch
    starts = list(range(0, len(order), bs))
    devs = gen_devices()[:max(1, len(starts))]
    main_dev = next(GEN_MODEL.parameters()).device
    replicas = [(GEN_MODEL if d == main_dev else copy.deepcopy(GEN_MODEL).to(d).eval(), d) for d in devs]
    work = queue.Queue()
    for b in starts: work.put(b)
    out, errors, lock = {}, [], threading.Lock()
    t0 = t_print = time.time(); done = [0]

    def worker(net, dev):
        try:
            while True:
                try: b = work.get_nowait()
                except queue.Empty: return
                chunk = order[b:b + bs]
                texts = generate_batch([enc[k] for k in chunk], CONFIG["gen_words"], CONFIG["max_new_tokens"],
                                       dec["temperature"], dec["top_k"], dec["top_p"],
                                       dec.get("repetition_penalty", 1.0), CONFIG["gen_ctx_tokens"],
                                       seed=CONFIG["seed"] + b, net=net, dev=dev)
                with lock:
                    for k, t in zip(chunk, texts): out[k] = {"text": t}
                    done[0] += 1
        except Exception as e:                                  # surface thread errors in the main thread
            errors.append(e)

    threads = [threading.Thread(target=worker, args=r, daemon=True) for r in replicas]
    for th in threads: th.start()
    if len(starts) > 1:
        print(f"  generating {len(order)} prompts in {len(starts)} batches on {len(devs)} device(s): "
              + ", ".join(map(str, devs)))
    last = -1
    while any(th.is_alive() for th in threads):
        time.sleep(2)
        with lock: n = done[0]
        if n != last and (n % log_every == 0 or time.time() - t_print > log_secs):
            if n > 0:
                eta = (time.time() - t0) / n * (len(starts) - n) / 60
                print(f"  {min(n * bs, len(order))}/{len(order)} done ({time.time()-t0:.0f}s, ETA {eta:.0f} min)")
            t_print, last = time.time(), n
    for th in threads: th.join()
    if errors: raise errors[0]
    del replicas
    if _use_cuda: torch.cuda.empty_cache()
    return {k: out[k] for k in ids}

GEN_MODEL = model                                # replaced by an ensemble in cell 9d if that scores better
DECODE = dict(temperature=CONFIG["temperature"], top_k=CONFIG["top_k"], top_p=CONFIG["top_p"],
              repetition_penalty=CONFIG["repetition_penalty"])
demo = generate_all({"d": {"text": "In the Supreme Court of India, the appellant contended that"}}, DECODE)
print(demo["d"]["text"])

In [ ]:
# 9b · Metrics + (optional) decoding search on dev prompts BEFORE the test set ---------
_BS = {}
def score_predictions(preds, refs, bertscore=CONFIG["run_bertscore"]):
    ids = list(refs)
    hyps = [preds[k]["text"] for k in ids]; gold = [refs[k]["text"] for k in ids]
    res = {}
    try:
        from rouge_score import rouge_scorer
        rs = rouge_scorer.RougeScorer(["rougeL"])
        res["rougeL_f1"] = float(np.mean([rs.score(g, h)["rougeL"].fmeasure for g, h in zip(gold, hyps)]))
    except Exception as e: print("ROUGE skipped:", e)
    try:
        import sacrebleu
        res["bleu4"] = sacrebleu.corpus_bleu(hyps, [gold]).score / 100
    except Exception as e: print("BLEU skipped:", e)
    if bertscore:
        try:
            if "scorer" not in _BS:                    # load roberta-large once, reuse for every call
                from bert_score import BERTScorer
                _BS["scorer"] = BERTScorer(lang="en", batch_size=16)
            _, _, F1 = _BS["scorer"].score(hyps, gold)
            res["bertscore_f1"] = float(F1.mean())
            if _use_cuda: torch.cuda.empty_cache()
        except Exception as e: print("BERTScore skipped:", e)
    return res

if CONFIG["decode_search"]:
    # balanced subset: the same number of dev prompts from each source
    by_src = {}
    for k, v in dev_refs.items(): by_src.setdefault(v.get("source", "all"), []).append(k)
    per = max(1, CONFIG["decode_search_n"] // len(by_src))
    sub_ids = [k for ks in by_src.values() for k in ks[:per]]
    sub_p = {k: dev_prompts[k] for k in sub_ids}; sub_r = {k: dev_refs[k] for k in sub_ids}
    search = []
    print(f"Decoding search on {len(sub_ids)} dev prompts:")
    for d in CONFIG["decode_grid"]:
        t0 = time.time()
        sc = score_predictions(generate_all(sub_p, d, log_every=10**9, log_secs=10**9), sub_r,
                               bertscore=CONFIG["run_bertscore"])
        search.append({**d, **sc, "sec": round(time.time() - t0)})
        print(f"  {d['name']:<20}", {k: round(v, 4) for k, v in sc.items()}, f"({time.time()-t0:.0f}s)")
    key = "bertscore_f1" if all("bertscore_f1" in r for r in search) else "rougeL_f1"
    best = max(search, key=lambda r: r.get(key, -1))
    DECODE = {k: best[k] for k in ("temperature", "top_k", "top_p", "repetition_penalty")}
    json.dump({"metric": key, "trials": search, "chosen": best["name"]},
              open(os.path.join(RUN_DIR, "decode_search.json"), "w"), indent=1)
    print(f"-> best by {key}: {best['name']}  -> used for the test set: {DECODE}")
else:
    print("Decoding search off; using CONFIG decoding:", DECODE)

In [ ]:
# 9d · Ensemble: average the next-token probabilities of several of your trained models -----
# p_ens(next | text) = sum_i w_i * p_i(next | text). All members share the tokenizer, so their
# distributions live on the same vocabulary. Each member keeps its own KV cache.
import types

class EnsembleLM(nn.Module):
    def __init__(self, members, weights):
        super().__init__()
        self.members = nn.ModuleList(members)
        w = torch.tensor(weights, dtype=torch.float32)
        self.register_buffer("logw", (w / w.sum()).log(), persistent=False)
        self.cfg = types.SimpleNamespace(max_seq_len=min(m.cfg.max_seq_len for m in members))

    def _mix(self, logits):                      # log of the weighted average of the probabilities
        lp = torch.stack([F.log_softmax(l.float(), dim=-1) for l in logits])          # (M,B,V)
        return torch.logsumexp(lp + self.logw.to(lp.device).view(-1, 1, 1), dim=0)

    @torch.no_grad()
    def prefill(self, prompt):
        outs = [m.prefill(prompt) for m in self.members]
        return self._mix([o[0] for o in outs]), [o[1] for o in outs]

    @torch.no_grad()
    def decode_step(self, token, caches, pos):
        outs = [m.decode_step(token, c, pos) for m, c in zip(self.members, caches)]
        return self._mix([o[0] for o in outs]), [o[1] for o in outs]

ARCH_CLASSES = {"prefixlm": (PrefixLMConfig, PrefixLM), "mla": (MLAConfig, MLALM)}

def load_member(path):
    st = torch.load(path, map_location=device, weights_only=False)
    arch = (st.get("config") or {}).get("arch", "decoder")
    assert arch in ARCH_CLASSES, f"{path}: arch={arch!r} has no KV cache; only prefixlm / mla can be ensembled"
    assert st.get("tok_hash") == TOK_HASH, f"{path}: different tokenizer ({st.get('tok_hash')} vs {TOK_HASH})"
    C, M = ARCH_CLASSES[arch]
    m = M(C(**st["cfg"])).to(device).eval()
    m.load_state_dict(st["model"])
    print(f"  member {arch:<9} {sum(p.numel() for p in m.parameters())/1e6:.1f}M params | step {st.get('step')} "
          f"| val loss {st.get('val_loss', float('nan')):.3f} | {path}")
    return m

ENSEMBLE = None
if CONFIG["ensemble_ckpts"]:
    assert hasattr(model, "prefill"), "the main model must be a prefixlm / mla model to ensemble"
    print("Ensemble members:")
    others = [load_member(pth) for pth in CONFIG["ensemble_ckpts"]]
    total = sum(p.numel() for p in model.parameters()) + sum(p.numel() for o in others for p in o.parameters())
    print(f"  combined parameters: {total/1e6:.1f}M (Phase-1 limit 300M)")
    by_src = {}
    for k, v in dev_refs.items(): by_src.setdefault(v.get("source", "all"), []).append(k)
    per = max(1, CONFIG["decode_search_n"] // len(by_src))
    ids = [k for ks in by_src.values() for k in ks[:per]]
    sp, sr = {k: dev_prompts[k] for k in ids}, {k: dev_refs[k] for k in ids}
    trials = [("single", None)] + [(f"ens{w}", w) for w in CONFIG["ensemble_weight_grid"]]
    results = []
    for name, w in trials:
        t0 = time.time()
        GEN_MODEL = model if w is None else EnsembleLM([model] + others, w)
        sc = score_predictions(generate_all(sp, DECODE, log_every=10**9, log_secs=10**9), sr)
        results.append((name, w, sc))
        print(f"  {name:<18}", {k: round(v, 4) for k, v in sc.items()}, f"({time.time()-t0:.0f}s)")
    key = "bertscore_f1" if all("bertscore_f1" in r[2] for r in results) else "rougeL_f1"
    best_name, best_w, _ = max(results, key=lambda r: r[2].get(key, -1))
    GEN_MODEL = model if best_w is None else EnsembleLM([model] + others, best_w)
    ENSEMBLE = best_w
    json.dump({"metric": key, "trials": [{"name": n, "weights": w, **sc} for n, w, sc in results],
               "chosen": best_name}, open(os.path.join(RUN_DIR, "ensemble_check.json"), "w"), indent=1)
    print(f"-> best by {key}: {best_name} -> used for the test set")
    if _use_cuda: torch.cuda.empty_cache()
print("Generation model:", "ensemble " + str(ENSEMBLE) if ENSEMBLE else type(model).__name__)

In [ ]:
# 9c · MBR (Minimum Bayes Risk) decoding: sample N, keep the most "central" candidate ---------
# Utility = ROUGE-L F1 between our OWN samples (purely lexical: no pretrained model is used to choose).
# The candidate with the highest average agreement with the other samples is the consensus answer.
def _lcs(a, b):
    if len(a) < len(b): a, b = b, a
    prev = [0] * (len(b) + 1)
    for x in a:
        cur = [0]
        for j, y in enumerate(b):
            cur.append(prev[j] + 1 if x == y else (prev[j + 1] if prev[j + 1] > cur[j] else cur[j]))
        prev = cur
    return prev[-1]

def _rouge_l(a, b):
    l = _lcs(a, b)
    return 0.0 if l == 0 else 2 * l / (len(a) + len(b))       # = 2PR/(P+R) with P=l/|a|, R=l/|b|

def _mbr_pick(texts):
    toks = [t.lower().split() for t in texts]
    n = len(toks); score = [0.0] * n
    for i in range(n):
        for j in range(i + 1, n):
            u = _rouge_l(toks[i], toks[j]); score[i] += u; score[j] += u
    full = [i for i in range(n) if len(toks[i]) >= CONFIG["gen_words"]] or list(range(n))
    return max(full, key=lambda i: score[i])                   # prefer complete 128-word candidates

def generate_mbr(prompt_dict, dec, n):
    cands, seed0 = [], CONFIG["seed"]
    for j in range(n):
        CONFIG["seed"] = seed0 + 7919 * j
        try:
            cands.append(generate_all(prompt_dict, dec, log_every=10**9))
        finally:
            CONFIG["seed"] = seed0
        print(f"  MBR sample {j + 1}/{n} done")
    keys = list(prompt_dict)
    groups = [[c[k]["text"] for c in cands] for k in keys]
    t0 = time.time(); picks = None
    if os.name == "posix":                                     # parallel on Kaggle (fork); sequential fallback
        try:
            from multiprocessing import Pool
            with Pool(os.cpu_count() or 2) as pool:
                picks = pool.map(_mbr_pick, groups, chunksize=16)
        except Exception as e:
            print("  (parallel MBR selection failed, running sequentially)", e)
    if picks is None:
        picks = [_mbr_pick(g) for g in groups]
    print(f"  MBR selection for {len(keys)} prompts: {time.time()-t0:.0f}s")
    return {k: {"text": g[i]} for k, g, i in zip(keys, groups, picks)}

USE_MBR = CONFIG["mbr_samples"] > 1
if USE_MBR and CONFIG["mbr_auto"]:
    by_src = {}
    for k, v in dev_refs.items(): by_src.setdefault(v.get("source", "all"), []).append(k)
    per = max(1, CONFIG["decode_search_n"] // len(by_src))
    ids = [k for ks in by_src.values() for k in ks[:per]]
    sp, sr = {k: dev_prompts[k] for k in ids}, {k: dev_refs[k] for k in ids}
    t0 = time.time()
    single = score_predictions(generate_all(sp, DECODE, log_every=10**9), sr)
    mbr = score_predictions(generate_mbr(sp, DECODE, CONFIG["mbr_samples"]), sr)
    key = "bertscore_f1" if "bertscore_f1" in single and "bertscore_f1" in mbr else "rougeL_f1"
    USE_MBR = mbr.get(key, -1) > single.get(key, -1)
    json.dump({"single": single, "mbr": mbr, "metric": key, "use_mbr": USE_MBR, "n": CONFIG["mbr_samples"]},
              open(os.path.join(RUN_DIR, "mbr_check.json"), "w"), indent=1)
    print(f"Dev check on {len(ids)} prompts ({time.time()-t0:.0f}s):")
    print("  single sample:", {k: round(v, 4) for k, v in single.items()})
    print(f"  MBR (N={CONFIG['mbr_samples']}):  ", {k: round(v, 4) for k, v in mbr.items()})
    print("-> test set uses", f"MBR with N={CONFIG['mbr_samples']}" if USE_MBR else "a single sample (MBR not better)")
print("MBR decoding:", f"ON (N={CONFIG['mbr_samples']})" if USE_MBR else "off")

In [ ]:
# 10 · Generate a continuation for every official test prompt ----------------------
with open(test_path, encoding="utf-8") as f:
    test_data = json.load(f)
assert isinstance(test_data, dict), f"Expected an id->record dict, got {type(test_data)}"

def get_text_field(v):
    if isinstance(v, str): return v
    if isinstance(v, dict):
        for k in ("text", "content", "prompt", "body"):
            if isinstance(v.get(k), str): return v[k]
        for vv in v.values():
            if isinstance(vv, str): return vv
    return ""

test_prompts = {k: {"text": get_text_field(v)} for k, v in test_data.items()}
wl = [len(p["text"].split()) for p in test_prompts.values()]
print(f"{len(test_prompts)} test prompts | words min {min(wl)} mean {np.mean(wl):.0f} max {max(wl)}")
print("Prompt tail:", repr(next(iter(test_prompts.values()))["text"][-150:]))
print("Decoding:", DECODE)
t0 = time.time()
preds = generate_mbr(test_prompts, DECODE, CONFIG["mbr_samples"]) if USE_MBR else generate_all(test_prompts, DECODE)
print(f"Generated {len(preds)} continuations in {(time.time()-t0)/60:.1f} min.")
# retry the few continuations that came out short, with a different seed each time
for attempt in range(1, 4):
    short = [k for k, v in preds.items() if len(v["text"].split()) < CONFIG["gen_words"]]
    if not short: break
    print(f"retry {attempt}: {len(short)} short outputs")
    seed0 = CONFIG["seed"]; CONFIG["seed"] = seed0 + 1000 * attempt
    try:
        redo = generate_all({k: test_prompts[k] for k in short}, DECODE, log_every=10**9)
    finally:
        CONFIG["seed"] = seed0
    for k in short:
        if len(redo[k]["text"].split()) > len(preds[k]["text"].split()): preds[k] = redo[k]

In [ ]:
# 11 · Validate, write prediction.json, zip, re-check the zip -----------------------
assert set(preds) == set(test_data), "Test IDs must match exactly (no missing/extra)."
for k, v in preds.items():
    assert isinstance(v["text"], str) and v["text"].strip(), f"Empty continuation for id {k}"
wc = [len(v["text"].split()) for v in preds.values()]
print(f"Word counts: min {min(wc)}, max {max(wc)}, mean {np.mean(wc):.1f} (target {CONFIG['gen_words']})")
if min(wc) < CONFIG["gen_words"]:
    print(f"WARNING: {sum(w < CONFIG['gen_words'] for w in wc)} outputs are short -> raise max_new_tokens.")

pred_path = os.path.join(RUN_DIR, "prediction.json")
with open(pred_path, "w", encoding="utf-8") as f:
    json.dump(preds, f, ensure_ascii=False)
zip_path = os.path.join(RUN_DIR, f"submission_{RUN_NAME}.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(pred_path, arcname="prediction.json")               # MUST be at the zip root
with zipfile.ZipFile(zip_path) as z:
    assert z.namelist() == ["prediction.json"], z.namelist()
    reread = json.loads(z.read("prediction.json"))
    assert set(reread) == set(test_data) and all("text" in v for v in reread.values())
print("Zip verified ->", zip_path, " (upload this file to Codabench)")

## 12 · Local evaluation on the held-out dev set (runs after the submission is safely written)
Val loss/perplexity measure the LM. The leaderboard scores **generation** with BERTScore, ROUGE-L and BLEU-4. Record both, because the brief asks you to study how they relate.
Add your own decoding settings to `DECODING_TRIALS`. Choosing among them, and explaining why, is your analysis. Then put the chosen values in `CONFIG`.
*Note: this local BERTScore uses `bert-score`'s default English model. Use the official evaluation scripts (Drive link in Sec 18.1) to match Codabench exactly, and compare runs relative to each other.*

In [ ]:
# 12 · Local evaluation of the chosen decoding on all dev prompts ---------------------
DECODING_TRIALS = [
    dict(name="chosen", **{k: v for k, v in DECODE.items() if k != "name"}),
    # add your own decoding settings here, one dict per trial
]
if session_hours() > CONFIG["session_limit_hours"] - 0.4:
    print(f"Only {12 - session_hours():.1f} h left in the session -> skipping decoding trials (val loss only).")
    DECODING_TRIALS = []
val_loss, val_ppl = evaluate()
dev_results = {"val_loss": val_loss, "val_ppl": val_ppl, "val_by_source": evaluate_by_source(), "decoding": []}
print(f"Val loss {val_loss:.3f} (ppl {val_ppl:.1f})")
for s, r in dev_results["val_by_source"].items():
    print(f"  {s:>14}: val loss {r['val_loss']:.3f} | ppl {r['val_ppl']:.1f}")
dev_sources = sorted({v.get("source", "all") for v in dev_refs.values()})
for trial in DECODING_TRIALS:
    t0 = time.time()
    dev_preds = generate_all(dev_prompts, trial, log_every=10**9)
    scores = score_predictions(dev_preds, dev_refs)
    if len(dev_sources) > 1:                     # same metrics per source (BERTScore off here: slow)
        scores["by_source"] = {s: score_predictions(dev_preds, {k: v for k, v in dev_refs.items()
                                                                 if v.get("source") == s}, bertscore=False)
                               for s in dev_sources}
    dev_results["decoding"].append({**trial, **scores, "gen_sec": round(time.time() - t0, 1)})
    print(trial["name"], {k: (round(v, 4) if isinstance(v, float) else v) for k, v in scores.items()})
json.dump(dev_results, open(os.path.join(RUN_DIR, "dev_results.json"), "w"), indent=1)
if DECODING_TRIALS:
    k0 = next(iter(dev_refs))
    print("\nREF :", dev_refs[k0]["text"][:300]); print("PRED:", dev_preds[k0]["text"][:300])

In [ ]:
# 13 · Log the experiment + export everything Phases 2-7 will need ------------------
record = dict(run=RUN_NAME, smoke=SMOKE_TEST, time=time.strftime("%Y-%m-%d %H:%M"),
              params=n_params, non_embedding_params=n_params - n_emb,
              best_step=state["step"], best_val_loss=state["val_loss"],
              best_val_ppl=math.exp(min(20, state["val_loss"])), train_minutes=round(TRAIN_MINUTES, 1),
              dev=dev_results, decoding_used=DECODE, mbr_samples=CONFIG["mbr_samples"] if USE_MBR else 0,
              ensemble=dict(ckpts=CONFIG["ensemble_ckpts"], weights=ENSEMBLE) if ENSEMBLE else None, data=meta, config=CONFIG,
              test_word_counts=[int(min(wc)), int(max(wc))])
json.dump(record, open(os.path.join(RUN_DIR, "record.json"), "w"), indent=1, default=str)
with open(os.path.join(OUT_ROOT, "experiments.jsonl"), "a") as f:
    f.write(json.dumps(record, default=str) + "\n")

shutil.copytree("a3src", os.path.join(RUN_DIR, "code"), dirs_exist_ok=True)   # exact code used
shutil.copy(os.path.join(DATA_DIR, "legal_bpe.pkl"), RUN_DIR)
print("Run folder contents:")
for fn in sorted(os.listdir(RUN_DIR)):
    p = os.path.join(RUN_DIR, fn)
    print(f"  {fn:<40} {os.path.getsize(p)/1e6:8.1f} MB" if os.path.isfile(p) else f"  {fn}/")

In [ ]:
# 14 · Package everything worth keeping into ONE zip + download links -----------------
# Commit run: open the version's Output tab and download  <RUN_NAME>_bundle.zip  (one file).
# Interactive session: click the links printed below.
INCLUDE_CKPT_LAST = True         # True = also keep the resumable checkpoint (~3x bigger than ckpt_best.pt)
bundle = os.path.join(OUT_ROOT, f"{os.path.basename(RUN_DIR)}_bundle.zip")
skip = set() if INCLUDE_CKPT_LAST else {"ckpt_last.pt"}
with zipfile.ZipFile(bundle, "w") as z:
    for root, _, files in os.walk(RUN_DIR):
        for fn in files:
            if fn in skip or fn.endswith((".tmp", ".pyc")) or "__pycache__" in root: continue
            p = os.path.join(root, fn)
            comp = zipfile.ZIP_STORED if fn.endswith(".pt") else zipfile.ZIP_DEFLATED   # .pt barely compresses
            z.write(p, arcname=os.path.join(os.path.basename(RUN_DIR), os.path.relpath(p, RUN_DIR)), compress_type=comp)
    exp_log = os.path.join(OUT_ROOT, "experiments.jsonl")
    if os.path.exists(exp_log): z.write(exp_log, arcname="experiments.jsonl")
with zipfile.ZipFile(bundle) as z:
    names = z.namelist()
print(f"Bundle: {bundle}  ({os.path.getsize(bundle)/1e6:.0f} MB, {len(names)} files)")
for n in sorted(names): print("  ", n)

try:                                             # clickable links (interactive sessions only)
    from IPython.display import FileLink, display
    for p in (bundle, os.path.join(RUN_DIR, f"submission_{RUN_NAME}.zip")):
        display(FileLink(os.path.relpath(p, os.getcwd())))
except Exception as e:
    print("(no clickable links here - download from the Output tab)", e)

## Next steps
1. **Smoke run.** Check that the initial loss is ≈ ln(V), the val loss drops *below* ln(V), the word counts are exactly 128, and the zip verifies. Upload the smoke zip once only to test the Codabench flow (starter/smoke runs do not count toward the minimum).
2. **Real run of this architecture.** Set `SMOKE_TEST=False` and commit. Save the output as a Kaggle Dataset (it holds `data_bpe8000/` + `runs/<RUN_NAME>/`). This is **submission #1**.
3. **Architectures #2–#4 (your own design and code).** Copy this notebook, set `prebuilt_dir` to the saved `data_bpe8000`, add your model file under `a3src/`, and build it in cell 7. The rest of the pipeline only needs `model(idx) -> logits (B, T, V)`. Keep the data, validation windows and dev set the same so the comparison is fair. Use a separate Kaggle account or session per architecture to run them in parallel.
4. **Think ahead to Phase 2** (starts 7 Oct). It requires **all four** Phase-1 models, a token/span head (so each model should be able to return hidden states), and a knowledge-distillation experiment (larger teacher → smaller student). Keep every `ckpt_best.pt` + `code/` + `legal_bpe.pkl`.
5. **Report evidence.** `experiments.jsonl`, `history.json`, `loss_curves.png` and `dev_results.json` per run. The analysis (what worked, what failed, and why) has to be your own writing.